# Polaris API → VictoriaLogs log-coverage



**What this answers:** for every Polaris API, *if something went wrong on this endpoint
tomorrow, would there be a record of it?*

It is a **log-generating harness**, not an API client and not a functional test suite. Its
output is a statement about observability, not about Polaris correctness — and the answer is
"no" for ten endpoints before it starts. Demonstrating that concretely is the point.

Built from `log-coverage/PLAN-log-coverage.md`, which amends
`~/hynix/local-k8s/POLARIS-API-LOG-COVERAGE-NOTEBOOK.md`. Read the plan first; this notebook
does not restate it.

## Before you run this

```bash
kubectl -n logging       port-forward svc/vlsingle-victoria-logs-single-server 9428:9428
kubectl -n datahub-hynix port-forward deploy/fb-polaris-shipper 2020:2020   # metrics only
```

`local` env only. It creates and deletes a catalog, principals, roles, namespaces, tables and
views — **the cleanup DELETEs are themselves part of the test**, so cell 9 is not optional
tidying, it is a measurement.

## Guards this run needs, and why

| guard | why |
|---|---|
| **do not trip the HPA** | Polaris scales to 3 replicas at 80% CPU, all appending to one `polaris.log` (`local-k8s` #8). The ×20 loops below are two orders of magnitude below that. Replica count is recorded before and after; a change **invalidates the run**. |
| **watch the shipper pod** | rule 6's dedup state is per-pod and in memory. A restart mid-run re-logs the first hit per table, so "exactly one" stops being an invariant. |
| **never print a token or a secret** | notebooks get committed. |
| **do not change Polaris configuration** | `local-k8s` #11/#12: it works and ships continuously, and the `quarkus.log.file.enabled=false` discrepancy is recorded as *unexplained, not pursued*. |

## The expected column is not a table anyone typed

`log_coverage.Policy` extracts `polaris_noise_filter` out of the **deployed**
`logging/fb-values.yaml` and runs it under a real Lua interpreter. If that file or a Lua
binary is missing, cell 0 aborts and says the expected column is unavailable — there is
deliberately no Python re-implementation to fall back on.

## 0 — Preflight

Aborts loudly. A coverage matrix taken from a half-working pipeline is worse than none.

In [1]:
import pathlib, shutil, subprocess, sys, time
from datetime import datetime, timezone

_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from polaris_test_utils import *          # noqa: F401,F403
import polaris_test_utils as ptu
import vlogs
import log_coverage as lc

init_env("local")
require_not_prod("the log-coverage drive creates and deletes catalogs, principals and roles")

RUN = str(int(time.time()))
NB_PRINCIPAL, NB_PROLE = f"nb_{RUN}_principal", f"nb_{RUN}_prole"
STARTED = datetime.now(timezone.utc)
V = vlogs.VLogs(ptu.VLOGS_URL)
PRE, FAIL = {}, []


def sh(*args):
    """kubectl, when it is there. Returns stdout or None -- never raises."""
    if not shutil.which(args[0]):
        return None
    try:
        r = subprocess.run(args, capture_output=True, text=True, timeout=20)
        return r.stdout.strip() if r.returncode == 0 else None
    except Exception:
        return None


try:
    ROOT_TOK = root_token()
    PRE["polaris"] = f"OK   {ptu.POLARIS_URL}"
except Exception as exc:
    ROOT_TOK = None
    FAIL.append(f"Polaris: {type(exc).__name__}: {exc}")

ok, detail = V.ping()
PRE["victorialogs"] = f"{'OK  ' if ok else 'DOWN'} {ptu.VLOGS_URL}  ({detail})"
if not ok:
    FAIL.append(
        f"VictoriaLogs unreachable at {ptu.VLOGS_URL} -- "
        "kubectl -n logging port-forward svc/vlsingle-victoria-logs-single-server 9428:9428"
    )

try:
    POLICY = lc.load_policy(ptu.FB_VALUES_PATH)
    PRE["retention policy"] = f"OK   {POLICY.source}  sha256={POLICY.sha256[:16]}"
except lc.PolicyUnavailable as exc:
    POLICY = None
    FAIL.append(f"expected column UNAVAILABLE: {exc}")

# IS THE POLICY IN THE FILE THE POLICY THAT IS RUNNING? This repo's most expensive
# failure is configuration written and never applied, and this notebook's first run
# (2026-09-04) found exactly that: fb-values.yaml gained polaris_noise_filter at
# 08:26:18Z on 2026-09-03 and the shipper pod had been up since 08:04:06Z, so the
# parser was live, the policy was not, and NOTHING was dropped. Check before driving,
# rather than inferring it from the shape of the results afterwards.
ALLOW_UNDEPLOYED_POLICY = False      # set True to drive anyway, deliberately
POLICY_LIVE, POLICY_LIVE_DETAIL = None, "policy not loaded"
if POLICY is not None:
    _cm = sh("kubectl", "-n", "datahub-hynix", "get", "cm",
             "-l", "app.kubernetes.io/instance=fb-polaris-shipper", "-o", "yaml")
    POLICY_LIVE, POLICY_LIVE_DETAIL = lc.deployed_policy_status(_cm, POLICY)
PRE["policy DEPLOYED?"] = {True: "YES ", False: "NO  ", None: "??  "}[POLICY_LIVE] + POLICY_LIVE_DETAIL.splitlines()[0]
if POLICY_LIVE is False and not ALLOW_UNDEPLOYED_POLICY:
    FAIL.append(
        "THE RETENTION POLICY IS NOT DEPLOYED. A coverage matrix taken now is a "
        "statement about a pipeline nobody is running.\n    " + POLICY_LIVE_DETAIL
    )

# Do numeric LogsQL filters actually match? `fb-values.yaml` sets `type_int_key
# http_status response_size` so they should. This cannot be answered by looking at a
# returned record's JSON type -- VictoriaLogs hands every field back as a string
# whatever it indexed -- so ask the operational question instead.
NUMERIC_STATUS, NUMERIC_DETAIL = (None, "VictoriaLogs unreachable")
if ok:
    # AN INSTRUMENT CHECK, NOT A RUN STATISTIC. It has to look across the whole
    # retention slice to find any 4xx at all -- the run has not happened yet -- so
    # the detail carries its scope, and cell 7 takes the run-scoped reading. Run 1
    # printed `http_status:"404" -> 138` in the Validity table for a run that made
    # 23 of them, which reconciled with nothing else on the page.
    NUMERIC_STATUS, NUMERIC_DETAIL = vlogs.numeric_status_filters_work(
        V, since="24h", scope="cluster-wide, last 24h")
PRE["numeric status filters"] = (
    {True: "WORK  ", False: "BROKEN", None: "?     "}[NUMERIC_STATUS] + " " + NUMERIC_DETAIL
)

FB_BEFORE, FB_ERR = vlogs.fluentbit_metrics(ptu.FB_METRICS_URL)
# GUARDED. The first time this counter was ever read it returned 7,154,980,971,680 --
# four times the epoch in milliseconds, so the summed field was a timestamp. A number
# that reconciles with nothing on the page borrows the credibility of the ones that do.
_fb_before = vlogs.drop_records_total(FB_BEFORE)
PRE["fluent-bit drop_records"] = FB_ERR or (
    f"{_fb_before:,.0f} (before)" if abs(_fb_before) < vlogs.IMPLAUSIBLE_DROP_TOTAL
    else f"UNREADABLE ({_fb_before:,.0f}) -- {vlogs.drop_records_by_plugin(FB_BEFORE)}")

SHIPPER_BEFORE = sh(
    "kubectl", "-n", "datahub-hynix", "get", "pods",
    "-l", "app.kubernetes.io/instance=fb-polaris-shipper",
    "-o", "jsonpath={range .items[*]}{.metadata.name}@{.status.startTime}{end}",
) or "kubectl unavailable -- a shipper restart mid-run will NOT be detectable"
HPA_BEFORE = sh(
    "kubectl", "-n", "datahub-hynix", "get", "hpa",
    "-o", "jsonpath={range .items[*]}{.metadata.name}={.status.currentReplicas}{end}",
) or "kubectl unavailable -- HPA scaling will NOT be detectable"
PRE["shipper pod (before)"] = SHIPPER_BEFORE
PRE["polaris replicas (before)"] = HPA_BEFORE

for k, v in PRE.items():
    print(f"  {k:26} {v}")
if FAIL:
    raise RuntimeError(
        "PREFLIGHT FAILED:\n  - " + "\n  - ".join(FAIL)
    )
print(f"\n  run={RUN}   principal={NB_PRINCIPAL}   started={STARTED.isoformat()}")

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
  polaris                    OK   http://192.168.139.2:8181
  victorialogs               OK   http://localhost:9428  (HTTP 200)
  retention policy           OK   /Users/kade/hynix/local-k8s/logging/fb-values.yaml  sha256=063c184df3f9f475
  policy DEPLOYED?           YES the running ConfigMap carries this exact script
  numeric status filters     WORK   http_status:>=400 -> 95;  http_status:"404" -> 54  [cluster-wide, last 24h]
  fluent-bit drop_records    7,154,980,971,680 (before)
  shipper pod (before)       fb-polaris-shipper-fluent-bit-

## 0b — The v3 gate

Cell 0 proves the running ConfigMap matches the **file**. It cannot prove the file is policy
v3, and it says nothing about the report. Both are aborts, and neither can be inferred from the
shape of the results afterwards — which is the mistake run 1 existed to stop repeating.

`WINDOW_SECONDS` is read here off the deployed script and every wait below derives from it.
**Nothing in this notebook hardcodes 1800**: redeploy at 30 for a fast run and the assertions
stay honest.


In [2]:
import re

GATE, GATE_FAIL = {}, []

# Comments must be stripped before grepping. The v3 script carries a comment saying the dedup
# machinery was REMOVED, and a naive search reads that as the machinery still being present --
# which it did on 2026-09-04, and cost a round trip.
_body = re.sub(r"--\[\[.*?\]\]", "", POLICY.script, flags=re.S)
_body = "\n".join(re.sub(r"--.*$", "", ln) for ln in _body.splitlines())
for marker, want in (("RESOURCE_PATTERNS", True), ("MGMT_PREFIX", True),
                     ("DEDUP_MAX_KEYS", False), ("_now_override", True)):
    here = marker in _body
    GATE[f"v3 marker {marker}"] = f"{'OK  ' if here == want else 'WRONG'} present={here} want={want}"
    if here != want:
        GATE_FAIL.append(f"{marker}: present={here}, policy v3 expects {want}")

# THE constant. Read, never assumed -- and `Policy.report_windows` refuses a caller that
# disagrees with it, because the filter indexes windows with its own value and a mismatch
# crosses no boundary, returns no report, and looks like an empty result rather than an error.
WINDOW_SECONDS = POLICY.window_seconds
TICK_SECONDS = POLICY.tick_seconds
GATE["WINDOW_SECONDS (deployed)"] = f"{WINDOW_SECONDS}s"
GATE["dummy Interval_Sec"] = f"{TICK_SECONDS}s" if TICK_SECONDS else "NOT FOUND in config.inputs"

# The tick rate is not the report period -- but it bounds two things the period cannot. It
# bounds the STARTUP BLIND SPOT (records before the first tick are counted into no window), and
# at tick >= window a late tick moves the window index by TWO: the filter reports the window it
# was holding and the one in between never existed. No single record shows that; only a gap
# between consecutive window_starts does, which is what cell 14 looks for.
if TICK_SECONDS:
    _per = WINDOW_SECONDS / TICK_SECONDS
    GATE["ticks per window"] = f"{_per:.0f}"
    if TICK_SECONDS >= WINDOW_SECONDS:
        GATE_FAIL.append(
            f"Interval_Sec {TICK_SECONDS} >= WINDOW_SECONDS {WINDOW_SECONDS}: a late tick "
            "skips a whole window, and the skipped window's records are attributed to the "
            "previous one. Lower the dummy INPUT's Interval_Sec.")
    elif _per < 4:
        GATE["ticks per window"] += "   WARN: little margin for scheduling jitter"

REPORTS_SEEN = V.count(vlogs.app_report(), start="24h")
GATE["report records (24h)"] = REPORTS_SEEN
if REPORTS_SEEN == 0:
    GATE_FAIL.append("no report records at all -- the polaris.report tick is not reaching "
                     "the filter, or the report OUTPUT is not wired to VictoriaLogs")

TICK_LEAK = V.count(vlogs.tick_leak(), start="24h")
GATE["raw tick leak (want 0)"] = TICK_LEAK
if TICK_LEAK:
    GATE_FAIL.append(f"{TICK_LEAK} raw ticks reached VictoriaLogs -- the filter should swallow "
                     "every tick inside a window; this is ~2,880 junk records a day")

# THE GATE. A window with access_seen == 0 has no resource or principal rows to split INTO, so
# it cannot answer the question -- that is not a failure, it is an unproven gate, and the run
# below produces the traffic that settles it.
BUSY = V.latest_summary(with_traffic=True, since="24h")
if BUSY is None:
    GATE["array split"] = (f"UNPROVEN -- no window with traffic yet; next boundary in "
                           f"{V.seconds_to_boundary(WINDOW_SECONDS):.0f}s. Cell 11 settles it.")
else:
    _ws = BUSY["window_start"]
    SPLIT = V.report_types(window_start=_ws, since="24h")
    GATE["array split"] = f"{SPLIT}  (window {_ws})"
    if sorted(SPLIT) != ["principal", "resource", "summary"]:
        GATE_FAIL.append(
            f"the filter's array return did not split into three report_types: {SPLIT}. "
            "One record carrying numeric-keyed fields means the schema these tests target "
            "does not exist, and the fix is in local-k8s/logging/fb-values.yaml, not here.")
    else:
        _bad = lc.check_invariants(V.report_window(_ws, since="24h"))
        GATE["stored window invariants"] = "OK" if not _bad else _bad
        if _bad:
            GATE_FAIL.append("a STORED window violates its own schema: " + "; ".join(_bad))

for k, v in GATE.items():
    print(f"  {k:28} {v}")
if GATE_FAIL:
    raise RuntimeError("v3 GATE FAILED:\n  - " + "\n  - ".join(GATE_FAIL))


  v3 marker RESOURCE_PATTERNS  OK   present=True want=True
  v3 marker MGMT_PREFIX        OK   present=True want=True
  v3 marker DEDUP_MAX_KEYS     OK   present=False want=False
  v3 marker _now_override      OK   present=True want=True
  WINDOW_SECONDS (deployed)    30s
  dummy Interval_Sec           5s
  ticks per window             6
  report records (24h)         333
  raw tick leak (want 0)       0
  array split                  {'summary': 1, 'resource': 43, 'principal': 4}  (window 2026-09-07T01:38:30Z)
  stored window invariants     OK


## 1 — Which correlation key does this run get?

`logging.requestIdHeaderName: Polaris-Request-Id` **[from values]** says the client can supply
the request id, which would give exact per-call correlation across the access log *and* the
application log lines the request produced. Whether Polaris honours it is **[assumed]** and is
the one thing the plan could not settle from the repo.

Settle it in minute one, not hour three. If it fails, everything below still runs — the
fallback is `user_principal_name`, `%u` from the access log — but the matrix becomes per-run
rather than per-call, and the report has to say so.

In [3]:
from polaris_rest import PolarisREST

probe_rid = lc.request_id(RUN, 0, "probe")
adm_pc = PolarisREST(ptu.POLARIS_URL, ptu.REALM, token=ROOT_TOK,
                     extra_headers={"Polaris-Request-Id": probe_rid})

# GET /v1/catalogs is rule 7 ("everything else -> keep"), so it is stored on every call --
# the cheapest possible probe that is guaranteed not to be deduplicated or dropped.
r = adm_pc.list_catalogs()
echoed = r.headers.get("Polaris-Request-Id")
adm_pc.extra_headers = {}

poll = V.poll_until(
    vlogs.and_(vlogs.app_polaris(), vlogs.field_eq("mdc.requestId", probe_rid)),
    expect_at_least=1, timeout=45, interval=1, start="15m",
)
CORRELATION = "exact" if poll.satisfied else "fallback"

print(f"  sent     Polaris-Request-Id: {probe_rid}")
print(f"  echoed   Polaris-Request-Id: {echoed}   ({'honoured' if echoed == probe_rid else 'REWRITTEN by the server'})")
print(f"  found in VictoriaLogs: {len(poll)} record(s) after {poll.waited_s:.1f}s")
print(f"\n  CORRELATION MODE = {CORRELATION.upper()}")
if CORRELATION == "fallback":
    print("  -> mdc.requestId does not round-trip. Per-call rows become per-endpoint rows;")
    print("     the run is selected by user_principal_name and the time window instead.")
    print("     This is a finding in its own right -- record it in the report.")
INGEST_LAG_S = poll.waited_s

  sent     Polaris-Request-Id: nb-1788745242-000-probe
  echoed   Polaris-Request-Id: nb-1788745242-000-probe   (honoured)
  found in VictoriaLogs: 8 record(s) after 2.0s

  CORRELATION MODE = EXACT


## 2 — The denominator: three sources, kept apart

`PLAN` §2.2. Polaris 8181/8182 serves **no** OpenAPI document (measured 2026-08-31), so the
spec has to be vendored — run `./fetch_specs.sh` once. The three sources are unequal and are
never collapsed into a percentage:

| source | what it proves | what it cannot |
|---|---|---|
| **spec** | this VERSION names the endpoint | that this BUILD serves it — generic tables and policies are feature-flagged |
| **captured** | this DEPLOYMENT served it | anything about what was never called: a call log is a lower bound |
| **driven** | this notebook calls it | — |

In [4]:
import api_surface as surf
from iceberg_rest import build_create_table_payload, build_scan_report, build_schema

HERE = pathlib.Path.cwd()
SPEC, SPEC_FILES = lc.spec_inventory(HERE / "spec")
CAPTURED, CAPTURED_FROM = lc.captured_inventory(
    HERE.parent / "diagnostics" / "api-sql-profile" / "reports"
)

if not SPEC:
    print("  spec      NOT VENDORED -- run ./fetch_specs.sh. The 'candidate' column will be")
    print("            absent, and an endpoint this build does not serve cannot be told")
    print("            apart from one nobody has called.")
else:
    for f in SPEC_FILES:
        print(f"  spec      {f['file']:38} {f['paths']:3} paths  sha256={f['sha256'][:12]}")
    print(f"  spec      {len(SPEC)} operations")
print(f"  captured  {len(CAPTURED)} operations from {CAPTURED_FROM.name if CAPTURED_FROM else 'NOTHING -- no parsable matrix report'}")
print(f"  driven    43 operations (api_surface.operations)")

# The verdict is printed here, before anything is driven, because "confirmed gap" is a
# fact about the harness and does not need this run to establish it.
import collections as _c

DENOM = lc.coverage_rows(SPEC, CAPTURED, lc.driven_inventory(surf.operations(
    surf.SurfaceContext(), build_create_table_payload, build_scan_report)))
print("\n  " + "  ".join(f"{k}={v}" for k, v in
                          _c.Counter(r["verdict"] for r in DENOM).most_common()))
_gaps = [r for r in DENOM if r["verdict"] == "confirmed gap"]
if _gaps:
    print("  CONFIRMED GAPS -- this cluster served these and the notebook does not call them:")
    for r in _gaps[:20]:
        print(f"    {r['method']:6} {r['path']}")

  spec      polaris-management-service.yml          17 paths  sha256=f5d18f7310a2
  spec      rest-catalog-open-api.yaml              18 paths  sha256=91786328d812
  spec      63 operations
  captured  40 operations from doc-api-sql-matrix-admin-explained-20260903-142636.md
  driven    43 operations (api_surface.operations)

  driven=40  candidate=24


## 3 — Happy path, in dependency order

`api_surface.operations()` is the 43-operation surface, in issue order, and the order is
load-bearing: three chains depend on it (namespace create→drop, table create→rename→drop,
view create→rename→drop) and the management block creates a principal, a principal-role and a
catalog-role before granting on them and deleting them.

Driven with `log_coverage.drive_tagged`, **not** `api_surface.drive`: that one is built around
a SQL `Tracer` and aborts when the capture reads nothing, which is correct for the API→SQL
audit and meaningless here.

`elapsed_ms` is measured client-side because there is nowhere else to get it — the access-log
pattern has no `%D`, so **nothing in VictoriaLogs can say how long any request took**. These
numbers are the evidence for adding it.

In [5]:
from iceberg_rest import IcebergREST

adm_ic = IcebergREST(ptu.POLARIS_URL, ptu.REALM, token=ROOT_TOK)
SCHEMA = build_schema([(1, "id", "long", True), (2, "val", "string", False)])

# `apiprofile*` is the stem teardown_fixture sweeps, so keep it -- and build the
# fixture directly rather than via .stamped(), whose timestamp would not be this
# run's and would make a stranded catalog un-attributable.
FX = surf.ProbeFixture(prefix=f"apiprofile{RUN}")
surf.setup_fixture(FX, adm_pc, adm_ic, ptu.BUCKET, ptu.MINIO_ENDPOINT, SCHEMA,
                   build_create_table_payload)
print(f"  fixture ready: {FX.cat}")

IDENTITY, pc, ic, DRIVE_PRINCIPAL, IDENTITY_NOTES = lc.elect_drive_identity(
    adm_pc, adm_ic, ptu.POLARIS_URL, ptu.REALM, NB_PRINCIPAL, NB_PROLE, fixture=FX
)
for n in IDENTITY_NOTES:
    print(f"  identity  {n}")

# WHO ISSUED WHAT, MEASURED. The 43 operations do not all go out on the same
# client -- the management block uses ctx.adm, which is root -- so "the run's
# principal" is not a constant that can be written into the matrix. Run 1 wrote
# one into all 132 rows, which made the column an assertion about this notebook
# rather than an observation, and the per-principal margin unfalsifiable.
PRINCIPALS = lc.principal_registry([
    (adm_pc, "root"), (adm_ic, "root"),
    (pc, DRIVE_PRINCIPAL), (ic, DRIVE_PRINCIPAL),
])

CTX = FX.context(ic, pc, adm_ic, SCHEMA)
CTX.adm = adm_pc          # so the fixture's own calls are tagged too
OPS = surf.operations(CTX, build_create_table_payload, build_scan_report)

CALLS = lc.drive_tagged(OPS, CTX, RUN, principals=PRINCIPALS)
print(f"\n  drove {len(CALLS)} operations, elected identity {DRIVE_PRINCIPAL}")

import collections as _c
print("  issued by:", dict(_c.Counter(c.get("principal") or "?" for c in CALLS)))

import collections
print("  status distribution:", dict(collections.Counter(c["status"] for c in CALLS)))
_err = [c for c in CALLS if c["error"]]
if _err:
    print(f"  {len(_err)} HARNESS ERRORS (not refusals -- these are the harness failing):")
    for c in _err[:10]:
        print(f"    {c['label']}: {c['error']}")

  fixture ready: apiprofile1788745242_cat
  identity  driving as nb_1788745242_principal (scope=PRINCIPAL_ROLE:ALL)

  drove 43 operations, elected identity nb_1788745242_principal
  issued by: {'nb_1788745242_principal': 43}
  status distribution: {200: 22, 204: 11, 500: 2, 404: 1, 201: 6, 403: 1}


## 4 — Negative cases

Rule 3 (`http_status >= 400` → keep) is the pipeline's real safety net — **not** rule 1, since
a failed request is usually not an ERROR log line at all, just an access-log line with a 4xx
and nothing else. These drive it.

The 500 is `create_catalog_no_endpoint` (`error-cases/09_500_null_pointer`): a catalog missing
`endpointInternal` throws a NullPointerException on table create. Cheapest reliable 500 in
this repo, and reversible — the catalog is deleted immediately after. It is here to answer the
plan's highest-value unknown: **do exception stack traces survive the pipeline?**

The timeout case is honest about its own limit. A client-side timeout does not stop Quarkus
completing the response and writing the access-log line, so it cannot produce the blind spot
being described. The real blind spot is a **server** hang, which cannot be provoked from here.

In [6]:
import requests

NEG = []
seq = 900


def neg(label, fn, method="", path=""):
    global seq
    seq += 1
    row = lc.call_once([pc, ic, adm_pc, adm_ic], RUN, seq, label, fn, method, path,
                       principals=PRINCIPALS)
    NEG.append(row)
    print(f"  {row['status'] if row['status'] is not None else row['error'][:40]:>6}  {label}")
    return row


# 404 -- a table that is not there. Rule 3 keeps EVERY one of these; rule 6 never sees it.
neg("neg.404_missing_table", lambda: ic.load_table(FX.cat, FX.ns, "definitely_not_here"),
    "GET", "")

# 401 -- a real client id with the wrong secret. The token endpoint's SUCCESS is dropped by
# rule 5; its FAILURE is kept by rule 3. So "who failed to authenticate" is answerable and
# "who authenticated" is not.
neg("neg.401_bad_secret", lambda: adm_pc.get_token(ptu.ROOT_CLIENT, "not-the-secret"),
    "POST", "")

# 403 -- a principal with a role and no grants at all.
DENIED = f"nb_{RUN}_denied"
DENIED_ROLE = f"nb_{RUN}_denied_role"
denied_ic = None
try:
    _cid, _sec = lc.provision_run_principal(adm_pc, DENIED, DENIED_ROLE,
                                            fixture=None, service_admin=False)
    _r = adm_pc.get_token(_cid, _sec, scope=f"PRINCIPAL_ROLE:{DENIED_ROLE}")
    if _r.status_code == 200:
        denied_ic = IcebergREST(ptu.POLARIS_URL, ptu.REALM, token=_r.json()["access_token"])
        # The denied principal is a SECOND identity in the report, not just a 403:
        # register it so its rows are attributed to it and not to the run principal.
        PRINCIPALS.update(lc.principal_registry([(denied_ic, DENIED)]))
        # denied_ic must be tagged too. On 2026-09-04 it was not in the client list, so
        # the 403 went out with no Polaris-Request-Id, could not be found afterwards,
        # and the matrix reported "EXPECTED STORED, ABSENT" for a record that was
        # almost certainly there -- a harness gap wearing a finding's clothes.
        seq += 1
        row = lc.call_once([denied_ic], RUN, seq, "neg.403_no_privilege",
                           lambda: denied_ic.load_table(FX.cat, FX.ns, FX.tbl), "GET", "",
                           principals=PRINCIPALS)
        NEG.append(row)
        print(f"  {row['status']:>6}  neg.403_no_privilege")
    else:
        print(f"  ---     403 case skipped: token for {DENIED} -> {_r.status_code}")
except Exception as exc:
    print(f"  ---     403 case skipped: {type(exc).__name__}: {exc}")

# 409 -- create a namespace that already exists.
neg("neg.409_already_exists", lambda: ic.create_namespace(FX.cat, FX.ns), "POST", "")

# 400 -- a create-table body with no schema.
neg("neg.400_malformed_body", lambda: ic.create_table(FX.cat, FX.ns, {"name": "no_schema"}),
    "POST", "")

# 405 -- a method the endpoint does not implement.
neg("neg.405_method_not_allowed",
    lambda: requests.request("DELETE", f"{ptu.POLARIS_URL}/api/catalog/v1/config",
                             headers=ic._h(), timeout=15),
    "DELETE", "")

# 500 -- NullPointerException on table create against a catalog with no endpointInternal.
NPE_CAT = f"nb{RUN}npe"
try:
    _r = create_catalog_no_endpoint(name=NPE_CAT, token=ROOT_TOK)
    print(f"  {_r.status_code:>6}  neg.500_setup_catalog_without_endpointInternal")
    ic.create_namespace(NPE_CAT, "npe_ns")
    neg("neg.500_null_pointer",
        lambda: ic.create_table(NPE_CAT, "npe_ns",
                                build_create_table_payload("npe_tbl", SCHEMA)),
        "POST", "")
finally:
    try:
        adm_pc.delete_catalog(NPE_CAT, purge=True)
    except Exception as exc:
        print(f"  ---     could not remove {NPE_CAT}: {exc}")

# timeout -- attempted, and expected NOT to reproduce the blind spot. Recorded either way.
neg("neg.client_timeout",
    lambda: requests.get(f"{ptu.POLARIS_URL}/api/management/v1/catalogs",
                         headers=adm_pc._h(), timeout=0.001),
    "GET", "/api/management/v1/catalogs")

# 429 is UNREACHABLE: rateLimiter.type is no-op [from values]. Recorded as not callable,
# with the reason. An honest gap beats a fabricated pass.
NOT_CALLABLE = {
    "429 rate limited": "rateLimiter.type: no-op -- the limiter cannot produce a 429",
    "OPA authorization paths": "opa.enabled: false -- those code paths are dead here",
    "federated catalog endpoints": "no federated catalog is configured on this cluster",
    "a server-side hang": "cannot be provoked from a client; the access log is written when "
                          "the response is, so a hang leaves NO line at all and this pipeline "
                          "is blind to it",
}

     404  neg.404_missing_table
     401  neg.401_bad_secret
     403  neg.403_no_privilege
     409  neg.409_already_exists
     400  neg.400_malformed_body
     405  neg.405_method_not_allowed
     201  neg.500_setup_catalog_without_endpointInternal
     200  neg.500_null_pointer
  ReadTimeout: HTTPConnectionPool(host='19  neg.client_timeout


## 5 — Policy probes

Deliberate tests of the **v3** branches. Every expected count comes from `Policy.predict()` —
the deployed Lua, run in issue order — so the expected column cannot drift from what ships.

v2's probes asked how dedup behaved. **There is no dedup in v3**, so those questions have no
answer and have been replaced rather than left to pass vacuously. The interesting question is
now *counted or kept*, and the two halves of the v3 trade are probed explicitly: every
management mutation is **stored** (the audit hole v2 left open, closed), and every successful
read is **counted only** (what it cost).

The ordering in the first block is load-bearing: read the table successfully, **then** write
it, **then** provoke a 403 on it, so one resource row carries `requests`, `reads`, `writes` and
`errors` all non-zero. An error on a resource never read successfully lands in `__other__`
instead of creating a key.


In [7]:
PROBE_CAT, PROBE_NS, PROBE_TBL, PROBE_VIEW = FX.cat, FX.ns, FX.tbl, FX.view
PROBES, seq = [], 1000


def probe(name, n, fn, method, path_for_prediction):
    """Run `fn` n times, tagged. The expectation is computed in cell 8, not here.

    Predicted across the WHOLE run in issue order rather than per probe. Under
    v2 that mattered because rule 6 carried per-day state; under v3 the filter
    keeps no keep/drop state at all, but the REPORT counters do accumulate
    across the run, so the scope still has to be the run.
    """
    global seq
    rows = []
    for i in range(n):
        seq += 1
        # EVERY client that might issue the call has to be in this list, or the request goes
        # out with no Polaris-Request-Id and cannot be found afterwards -- which the matrix
        # then reports as "EXPECTED STORED, ABSENT", a harness gap wearing a finding's
        # clothes. That happened to the 403 case in run 1, was fixed for the negative cell,
        # and I reintroduced it here for the 403 PROBE. Measured 2026-09-04: stored 0,
        # app_lines 0, for a record that was certainly there.
        rows.append(lc.call_once(TAGGABLE, RUN, seq, f"probe.{name}",
                                 fn, method, path_for_prediction,
                                 principals=PRINCIPALS))
    PROBES.append({"probe": name, "calls": n, "rows": rows})
    codes = sorted({r["status"] for r in rows})
    print(f"  {name:46} calls={n:<3} status={codes}")
    return rows


TAGGABLE = [c for c in (pc, ic, adm_pc, adm_ic, denied_ic) if c is not None]
CATP = f"/api/catalog/v1/{PROBE_CAT}"
MGMTP = "/api/management/v1"
TBL_PATH = f"{CATP}/namespaces/{PROBE_NS}/tables/{PROBE_TBL}"

# --- one resource row with requests, reads, writes AND errors all non-zero ---
# ORDER IS LOAD-BEARING. Read it successfully FIRST: `touch_resource(key, kind, create)` is
# called with create=false for an error, so a 403 on a table nobody has read yet lands in
# __other__ and never creates the key. Read, then write, then fail.
probe("read the table (rule 6, counted)", 3,
      lambda: ic.load_table(PROBE_CAT, PROBE_NS, PROBE_TBL), "GET", TBL_PATH)
probe("write the table (rule 5b, counted; writes+1)", 1,
      lambda: ic.commit_table(PROBE_CAT, PROBE_NS, PROBE_TBL,
                              [{"action": "set-properties", "updates": {"nb.run": RUN}}]),
      "POST", TBL_PATH)
if denied_ic is not None:
    probe("403 on the table (rule 3, KEPT; errors+1)", 1,
          lambda: denied_ic.load_table(PROBE_CAT, PROBE_NS, PROBE_TBL), "GET", TBL_PATH)
else:
    print("  !! no unauthorized client -- the table row will carry errors=0 and cell 12's "
          "four-non-zero-fields assertion is VOID, not failed")
# --- PLAN 6.2: TWO principals, with DIFFERENT mixes -------------------------------
# With one principal the margin equality proves nothing. `sum(principal.requests)
# == access_seen - parse_errors` is satisfied identically by a global counter when
# there is only one row to sum, so a filter that attributed nothing at all would
# still pass -- and run 1 drove all 132 calls as one principal and reported the
# margin OK. Give the denied principal a read-heavy mix (its GETs all 403, so it
# carries reads AND errors) and the run principal a write-heavy one, so neither
# principal row can equal the run total and the equality has something to catch.
if denied_ic is not None:
    probe("denied principal reads x8 (403; the per-principal margin)", 8,
          lambda: denied_ic.load_table(PROBE_CAT, PROBE_NS, PROBE_TBL), "GET", TBL_PATH)
probe("run principal writes x6 (write-heavy mix)", 6,
      lambda: ic.commit_table(PROBE_CAT, PROBE_NS, PROBE_TBL,
                              [{"action": "set-properties",
                                "updates": {"nb.mix": RUN}}]),
      "POST", TBL_PATH)

# `/metrics` is driven by the happy path (api_surface's iceberg.report_metrics builds a real
# ScanReport). Cell 12 asserts what matters about it: its requests land on the TABLE's row and
# NO row exists for the /metrics path. v2 emitted two rows for one table.

# --- rule 6: successful reads are counted only. All of these were STORED under v2. ---
probe("same table GET x20 (counted)", 20,
      lambda: ic.load_table(PROBE_CAT, PROBE_NS, PROBE_TBL), "GET", TBL_PATH)
probe("HEAD the same table (counted)", 1,
      lambda: ic.head_table(PROBE_CAT, PROBE_NS, PROBE_TBL), "HEAD", TBL_PATH)
probe("table LIST x20 (counted; v2 stored 20)", 20,
      lambda: ic.list_tables(PROBE_CAT, PROBE_NS),
      "GET", f"{CATP}/namespaces/{PROBE_NS}/tables")
# get_config() WITHOUT a warehouse returns 400 on this build, so the bare call tests rule 3,
# not rule 6 -- run 1 recorded three 400s under a label promising a counted 2xx GET.
probe("GET /config (counted; v2 stored every one)", 3,
      lambda: ic.get_config(warehouse=PROBE_CAT),
      "GET", f"/api/catalog/v1/config?warehouse={PROBE_CAT}")
probe("management read (counted)", 1,
      lambda: adm_pc.list_principals(), "GET", f"{MGMTP}/principals")
# The happy path renames probe_view -> probe_view2 and drops it, so by the time the probes
# run there is no view left to read: this 404s. It still exercises rule 3 and the classifier
# still returns resource_kind=view, but it is NOT a successful view read -- that comes from
# the happy path's own load_view/head_view, which is where `view` in the kind list is earned.
probe("view read after the rename (404; classifies as view)", 1,
      lambda: ic.load_view(PROBE_CAT, PROBE_NS, PROBE_VIEW),
      "GET", f"{CATP}/namespaces/{PROBE_NS}/views/{PROBE_VIEW}")
probe("namespace read (resource_kind=namespace)", 1,
      lambda: ic.load_namespace(PROBE_CAT, PROBE_NS),
      "GET", f"{CATP}/namespaces/{PROBE_NS}")

# --- rule 3: errors are kept, ALL of them, no cap -- and they do not create a key ---
probe("missing table GET x20 (all KEPT, -> __other__)", 20,
      lambda: ic.load_table(PROBE_CAT, PROBE_NS, "nope"),
      "GET", f"{CATP}/namespaces/{PROBE_NS}/tables/nope")

# --- rule 5a: THE V2 AUDIT HOLE, now closed. Every one of these must be STORED. ---
ASYM = f"nb_{RUN}_asym"
probe("create a principal (rule 5a, now KEPT)", 1,
      lambda: adm_pc.create_principal(ASYM), "POST", f"{MGMTP}/principals")
probe("reset its credentials (rule 5a, now KEPT)", 1,
      lambda: adm_pc.reset_principal_credentials(ASYM),
      "POST", f"{MGMTP}/principals/{ASYM}/reset")
probe("delete that principal (rule 4)", 1,
      lambda: adm_pc.delete_principal(ASYM), "DELETE", f"{MGMTP}/principals/{ASYM}")

# --- rule 5b: every other successful POST is counted only ---
probe("successful token x3 (counted)", 3,
      lambda: adm_pc.get_token(ptu.ROOT_CLIENT, ptu.ROOT_SECRET),
      "POST", "/api/catalog/v1/oauth/tokens")
RENAMED = f"{PROBE_TBL}_renamed"
probe("rename a table (counted)", 1,
      lambda: ic.rename_table(PROBE_CAT, PROBE_NS, PROBE_TBL, PROBE_NS, RENAMED),
      "POST", f"{CATP}/tables/rename")
probe("rename it back (counted)", 1,
      lambda: ic.rename_table(PROBE_CAT, PROBE_NS, RENAMED, PROBE_NS, PROBE_TBL),
      "POST", f"{CATP}/tables/rename")


  read the table (rule 6, counted)               calls=3   status=[200]
  write the table (rule 5b, counted; writes+1)   calls=1   status=[200]
  403 on the table (rule 3, KEPT; errors+1)      calls=1   status=[403]
  denied principal reads x8 (403; the per-principal margin) calls=8   status=[403]
  run principal writes x6 (write-heavy mix)      calls=6   status=[200]
  same table GET x20 (counted)                   calls=20  status=[200]
  HEAD the same table (counted)                  calls=1   status=[204]
  table LIST x20 (counted; v2 stored 20)         calls=20  status=[200]
  GET /config (counted; v2 stored every one)     calls=3   status=[200]
  management read (counted)                      calls=1   status=[200]
  view read after the rename (404; classifies as view) calls=1   status=[404]
  namespace read (resource_kind=namespace)       calls=1   status=[200]
  missing table GET x20 (all KEPT, -> __other__) calls=20  status=[404]
  create a principal (rule 5a, now KEPT)       

[{'seq': 1094,
  'label': 'probe.rename it back (counted)',
  'method': 'POST',
  'path': '/api/catalog/v1/apiprofile1788745242_cat/tables/rename',
  'actual_path': '/api/catalog/v1/apiprofile1788745242_cat/tables/rename',
  'issued_at': 1788745249.3885992,
  'principal': 'nb_1788745242_principal',
  'api': 'catalog',
  'request_id': 'nb-1788745242-1094-probe-rename-it-back-counted',
  'status': 200,
  'elapsed_ms': 12.5,
  'error': None,
  'echoed_request_id': 'nb-1788745242-1094-probe-rename-it-back-counted'}]

## 6 — Where is a PUT's request body logged?

Rule 4 keeps every PUT/DELETE/PATCH, but **the request body is not in the access log**. A PUT
that changes a role's privileges is recorded as *"a PUT happened to this path, and it returned
200"*. For an audit trail of grants that is the whole point, and it is missing.

So: grant a privilege, then search VictoriaLogs for *any* record carrying the privilege name.
Then look at the other channel. The `eventListener` writes audit events to a PostgreSQL table
(`type: persistence-in-memory-buffer`, 5s flush, 1000-event buffer **[from values]**) — a
second audit channel that does not go through VictoriaLogs at all. If that is where "what
changed" lives, **it changes where you go during an incident**, and the report has to say so.

In [8]:
GRANTED_PRIV = "TABLE_READ_DATA"
seq += 1
grant_row = lc.call_once([pc, ic, adm_pc, adm_ic], RUN, seq, "probe.grant_privilege",
                         lambda: adm_pc.grant_privilege(FX.cat, "catalog_admin", GRANTED_PRIV),
                         "PUT", f"/api/management/v1/catalogs/{FX.cat}/catalog-roles/catalog_admin/grants",
                         principals=PRINCIPALS)
print(f"  grant {GRANTED_PRIV} -> {grant_row['status']}")

time.sleep(max(2.0, INGEST_LAG_S))
priv_hits = V.query(vlogs.and_(vlogs.app_polaris(), f"_msg:{vlogs.quote(GRANTED_PRIV)}"),
                    start="10m", limit=50)
print(f"  records in VictoriaLogs carrying the privilege NAME: {len(priv_hits)}")
if not priv_hits:
    print("  -> the pipeline cannot answer 'who granted what'. The PUT is stored; the body is not.")

# Polaris creates its objects in POLARIS_SCHEMA, not `public` -- looking there reported
# "no events table in this metastore" on 2026-09-04 for a table that may well exist.
EVENTS = {"reachable": False, "rows": None, "sample": None, "note": None, "schema": None}
try:
    import psycopg2
    with psycopg2.connect(**ptu.PG_CONFIG) as conn:
        EVENTS.update(lc.events_table(conn))
        EVENTS["reachable"] = EVENTS.get("rows") is not None
except Exception as exc:
    EVENTS["note"] = f"{type(exc).__name__}: {exc}"

if EVENTS["reachable"]:
    print(f"  found `events` in schema {EVENTS['schema']}")
    print(f"  eventListener `events` table: {EVENTS['rows']} rows -- a SECOND audit channel,")
    print("  and it does not pass through VictoriaLogs at all.")
    for row in EVENTS["sample"] or []:
        print("   ", {k: str(v)[:60] for k, v in row.items()})
else:
    print(f"  events table not read: {EVENTS['note']}")
    print("  (needs a port-forward: kubectl -n datahub-hynix port-forward "
          "pod/benchmarks-postgresql-postgresql-ha-postgresql-0 5433:5432)")

  grant TABLE_READ_DATA -> 201
  records in VictoriaLogs carrying the privilege NAME: 6
  found `events` in schema polaris_schema
  eventListener `events` table: 2194 rows -- a SECOND audit channel,
  and it does not pass through VictoriaLogs at all.
    {'realm_id': 'POLARIS', 'catalog_id': 'user51_catalog', 'event_id': 'cb94b83a-c67c-4f4a-a62a-8254892dc2d7', 'request_id': '3923fb9f-f6ae-4f2e-8c15-15420a28bf75_0000000000000000960', 'event_type': 'AfterCreateCatalogEvent', 'timestamp_ms': '1787533896559', 'principal_name': 'root', 'resource_type': 'CATALOG', 'resource_identifier': 'user51_catalog', 'additional_properties': '{}'}
    {'realm_id': 'POLARIS', 'catalog_id': 'user1_catalog', 'event_id': '3a6cd508-12f7-4622-b3f2-c13586cbe80c', 'request_id': '3923fb9f-f6ae-4f2e-8c15-15420a28bf75_0000000000000000968', 'event_type': 'AfterCreateCatalogEvent', 'timestamp_ms': '1787533961568', 'principal_name': 'root', 'resource_type': 'CATALOG', 'resource_identifier': 'user1_catalog', 'additiona

## 7 — Collection

Ingest is asynchronous, so this polls rather than sleeping and hoping — and it deduplicates by
`mdc.requestId` before counting, because the shipper's tail offset DB lives on an `emptyDir`:
a `helm upgrade` replaces the pod, `Read_from_Head true` re-reads the whole file, and
VictoriaLogs does not deduplicate on ingest. **Records can legitimately appear twice.**

In [9]:
ALL_CALLS = CALLS + NEG + [r for p in PROBES for r in p["rows"]] + [grant_row]
WINDOW_START = STARTED
WINDOW_END = datetime.now(timezone.utc)

if CORRELATION == "exact":
    run_filter = vlogs.and_(vlogs.app_polaris(), f'"mdc.requestId":"nb-{RUN}-"*')
else:
    run_filter = vlogs.and_(vlogs.app_polaris(),
                            vlogs.field_eq("user_principal_name", DRIVE_PRINCIPAL))

pull = V.settle(run_filter, quiet_for=4, timeout=90, interval=2,
                start=WINDOW_START, end=WINDOW_END, limit=10000)
# Deduplicate REPLAYED RECORDS, not request ids: one call produces many records that
# share a request id (its access-log line plus every application line it emitted), and
# keying on the request id collapses them into one. See vlogs.dedup_replayed.
STORED = vlogs.dedup_replayed(pull.records)
DUPES = len(pull.records) - len(STORED)

BY_RID = {}
for rec in STORED:
    BY_RID.setdefault(rec.get("mdc.requestId"), []).append(rec)

print(f"  calls issued          {len(ALL_CALLS)}")
print(f"  records returned      {len(pull.records)}  (settled after {pull.waited_s:.0f}s)")
print(f"  after replay dedup    {len(STORED)}   [{DUPES} duplicate(s) -- a shipper replay if > 0]")
# AGAINST ITS OWN DENOMINATOR. Run 1788745242 reported 147 distinct ids for 146 calls:
# the numerator counted every run-minted id in the pull, cell 1's correlation probe
# included, while the denominator counted the calls. Correlation is what the whole
# per-call matrix rests on, so the ratio has to be readable literally.
CORR = lc.correlation_stats(STORED, ALL_CALLS)
print(f"  request ids recovered {CORR['recovered']} of {CORR['with_id']} calls that carried one"
      f"   ({CORR['other_ids']} further run id(s) belong to calls outside this table)")
if CORR["missing"]:
    print(f"   NOT RECOVERED: {CORR['missing'][:6]}")
print(f"  records per call      {len(STORED) / max(1, len(ALL_CALLS)):.1f}")

# THE RUN'S OWN numeric-filter reading. Cell 0's is cluster-wide over 24h and
# answers "does type_int_key work at all"; this one answers "does it find THIS
# run's errors", which is the question a reader of the Validity table thinks is
# being answered. Both are reported, each labelled with its scope.
NUMERIC_RUN, NUMERIC_RUN_DETAIL = vlogs.numeric_status_filters_work(
    V, since=WINDOW_START, end=WINDOW_END, scope="this run")
print(f"  numeric filters (run) {NUMERIC_RUN_DETAIL}")

# WHERE EVERY STORED RECORD WENT. Run 1 reported 2,117 records while the matrix's
# own columns summed to 2,109: eight records attributed to nothing at all. Since
# app_lines is len(found) - len(access) from THIS pull, the two figures agree by
# construction or something is unattributed -- a record carrying a request id this
# run minted for a call that is not in ALL_CALLS, or no request id at all. Name
# both rather than leaving a residue for the reader to notice.
VOLUME = lc.reconcile_volume(STORED, ALL_CALLS, run=RUN)
print(f"  volume reconciles     {VOLUME['reconciles']}   attributed="
      f"{VOLUME['attributed']}  other run ids={VOLUME['run_other']}  "
      f"untagged={VOLUME['untagged']}  of {VOLUME['total']}")

FB_AFTER, FB_AFTER_ERR = vlogs.fluentbit_metrics(ptu.FB_METRICS_URL)
DROP_DELTA, DROP_NOTE = ((None, FB_ERR or FB_AFTER_ERR) if (FB_ERR or FB_AFTER_ERR)
                         else vlogs.drop_records_delta(FB_BEFORE, FB_AFTER))
print(f"  fluent-bit drop delta {DROP_DELTA if DROP_DELTA is not None else DROP_NOTE}")
# WAS ANYTHING LOST IN TRANSIT? The output counters are cumulative since the pod
# started, so a zero here is a statement about that pod's whole life. It is what
# separates "the filter never emitted a window" from "it emitted one and the HTTP
# output dropped it" -- the two candidates for the missing windows of 2026-09-07,
# which the report stream alone cannot tell apart.
FB_OUT, FB_OUT_ERR = vlogs.fluentbit_metrics(ptu.FB_METRICS_URL, section="output")
OUT_HEALTH = None if FB_OUT_ERR else vlogs.output_health(FB_OUT)
print(f"  output transit loss   {OUT_HEALTH if OUT_HEALTH else FB_OUT_ERR}")

if DROP_DELTA is None and not (FB_ERR or FB_AFTER_ERR):
    print(f"   per filter, after: {vlogs.drop_records_by_plugin(FB_AFTER)}")
    print("   -> curl the metrics endpoint and read its shape; parse_fluentbit_metrics is")
    print("      summing the wrong field. Until then this run has NO suppression figure.")

SHIPPER_AFTER = sh("kubectl", "-n", "datahub-hynix", "get", "pods",
                   "-l", "app.kubernetes.io/instance=fb-polaris-shipper",
                   "-o", "jsonpath={range .items[*]}{.metadata.name}@{.status.startTime}{end}")
HPA_AFTER = sh("kubectl", "-n", "datahub-hynix", "get", "hpa",
               "-o", "jsonpath={range .items[*]}{.metadata.name}={.status.currentReplicas}{end}")
SHIPPER_MOVED = bool(SHIPPER_AFTER) and SHIPPER_AFTER != SHIPPER_BEFORE
HPA_MOVED = bool(HPA_AFTER) and HPA_AFTER != HPA_BEFORE
if SHIPPER_MOVED:
    print("  !! THE SHIPPER RESTARTED MID-RUN. Dedup state is per-pod and in memory, so the")
    print("     'exactly one per day' expectations below are VOID for this run, not failed.")
if HPA_MOVED:
    print("  !! POLARIS SCALED MID-RUN. Multiple replicas append to one log file "
          "(local-k8s #8) -- this run is INVALID.")

  calls issued          146
  records returned      2500  (settled after 4s)
  after replay dedup    2500   [0 duplicate(s) -- a shipper replay if > 0]
  distinct request ids  147
  records per call      17.1
  numeric filters (run) http_status:>=400 -> 46;  http_status:"404" -> 25  [this run]
  volume reconciles     True   attributed=2492  other run ids=8  untagged=0  of 2500
  fluent-bit drop delta 52000.0


## 8 — The coverage matrix

One row per endpoint. `expected` comes from the deployed Lua; `stored` from VictoriaLogs; the
`discrepancy` column is the only one worth reading twice.

A row where expected is *drop* and stored is 0 is **not a pass** in the ordinary sense — it is
the pipeline working exactly as configured, and the configuration being wrong. Those rows are
the finding.

In [10]:
import pandas as pd

EXPECTED = lc.expected_for(ALL_CALLS, POLICY, user=DRIVE_PRINCIPAL) if POLICY else [None] * len(ALL_CALLS)

rows = []
for call, exp in zip(ALL_CALLS, EXPECTED):
    found = BY_RID.get(call["request_id"], []) if CORRELATION == "exact" else None
    access = [r for r in (found or []) if vlogs.is_access_log(r)]
    rows.append({
        "label": call["label"],
        "method": call["method"],
        # TRUNCATED FOR DISPLAY ONLY. Every statistic reads `path_full`: run 1
        # filtered this column for "/api/management" and reported 5 of 5
        # management POSTs kept, because the cut removed the prefix from
        # POST /api/management/v1/catalogs/{c}/catalog-roles. Six were kept.
        "path": (call["actual_path"] or call["path"] or "")[-58:],
        "path_full": lc.call_path(call),
        "status": call["status"],
        "ms": call["elapsed_ms"],
        "expected": (exp or {}).get("verdict") if exp else None,
        # `disposition` is the column v3 needs and v2 did not. `stored` is 0 for most calls
        # BY DESIGN now, so an expected/stored matrix mostly reads "correctly dropped" and
        # hides the interesting half: which aggregate row should have moved instead.
        "disposition": lc.disposition((exp or {}).get("verdict")) if exp else None,
        "stored": len(access) if found is not None else None,
        # app_lines is derived from THIS pull, not from a second source: `found` is every
        # record VictoriaLogs returned for the call's request id, `access` the access-log
        # subset, and the difference is the application lines the request emitted. That
        # settles the open question the previous run left ("where does app_lines come
        # from?") -- VictoriaLogs, via mdc.requestId, and nowhere else.
        "app_lines": len(found) - len(access) if found is not None else None,
    })
M = pd.DataFrame(rows)

# Which resource row and which principal row SHOULD each call have incremented? The keys come
# from the deployed classify() -- driven one path per window so the answer is the filter's own,
# never a Python re-implementation of RESOURCE_PATTERNS.
_paths = sorted({(c.get("actual_path") or lc.full_path(c["api"], c["path"])).split("?")[0]
                 for c in ALL_CALLS if c.get("status")})
KEYS = POLICY.classify_paths(_paths) if POLICY else {}
M["resource_row"] = [
    (KEYS.get((c.get("actual_path") or lc.full_path(c["api"], c["path"])).split("?")[0])
     or (None, None))[0] for c in ALL_CALLS
]
M["kind"] = [
    (KEYS.get((c.get("actual_path") or lc.full_path(c["api"], c["path"])).split("?")[0])
     or (None, None))[1] for c in ALL_CALLS
]
# MEASURED, from each request's own Authorization header -- see lc.principal_of.
# Run 1 wrote DRIVE_PRINCIPAL into all 132 rows; the management operations went
# out as root and the 403 as nb_<run>_denied, so the column agreed with itself
# and with nothing else.
M["principal_row"] = [c.get("principal") or "-" for c in ALL_CALLS]
# WHAT resource_row MEANS. It is the key this path CLASSIFIES to, which is where the call
# lands if it succeeds. An error on a resource that was never read successfully increments
# `__other__` instead -- errors pass create=false and never create a key. So for a 4xx/5xx
# row, read this column as "the key it would have hit", not as the row that moved.
print(f"\n  resource_row is the key the path classifies to. For the "
      f"{int((M.status.fillna(0) >= 400).sum())} error calls it is where the call WOULD have "
      "landed:\n  an error on a resource not already known increments __other__ instead.")
M["discrepancy"] = [
    "" if r.expected is None or r.stored is None
    else "" if (r.expected == "keep") == (r.stored > 0)
    else ("EXPECTED STORED, ABSENT" if r.expected == "keep" else "EXPECTED DROPPED, PRESENT")
    for r in M.itertuples()
]
pd.set_option("display.width", 200, "display.max_rows", 300, "display.max_colwidth", 60)
#: `path_full` is the column the statistics read and the one nobody can fit on a
#: terminal. Cell 11b adds `window_start` and `counted_where` once the report is in.
DISPLAY_DROP = ["path_full"]
print(M.drop(columns=DISPLAY_DROP).to_string(index=False))

print("\n--- who issued these calls (measured, not assumed) ---")
for _p, _n in sorted(M.principal_row.value_counts().items(), key=lambda kv: -kv[1]):
    print(f"  {str(_p):40} {_n:>4} calls")
if M.principal_row.nunique() < 2:
    print("  !! ONE identity drove everything: sum(principal.requests) is then the run")
    print("     total by construction and the margin equality proves nothing (PLAN 6.2).")

print("\n--- calls that leave NO individual record: COUNTED, not lost ---")
_drop = M[M.expected == "drop"]
print(f"  {len(_drop)} calls; {int((_drop.stored.fillna(0) > 0).sum())} of them are stored anyway")
print(_drop[["label", "method", "status", "stored", "resource_row", "kind"]].to_string(index=False))

print("\n--- the v3 trade, both halves ---")
MGMT_POSTS = lc.mgmt_post_stats(ALL_CALLS, EXPECTED)
print(f"  management POSTs driven: {MGMT_POSTS['driven']};  kept: {MGMT_POSTS['kept']}"
      f"   ({MGMT_POSTS['kept_2xx']} by the management-POST rule, {MGMT_POSTS['kept_error']}"
      " that a 4xx would have kept anyway --")
print("    first match wins, and the error rule is matched first. v2 kept NONE of these:")
print("    a principal was created invisibly and deleted visibly, and a credential reset")
print("    left no trace at all. That is the hole v3 closes.)")
for _l, _p, _s, _v in MGMT_POSTS["rows"]:
    print(f"    {str(_l)[:44]:44} {_s} {_v:5} {_p[-48:]}")
_reads = M[(M.method.isin(["GET", "HEAD"])) & (M.status.fillna(0) < 400)]
print(f"  successful reads driven: {len(_reads)};  counted only: "
      f"{int((_reads.expected == 'drop').sum())}   (what it cost: no per-call timestamp,")
print("    no per-table access frequency -- only the window aggregate.)")

print("\n--- resource_kind coverage, from the deployed classify() ---")
# isinstance, not truthiness: pandas stores a missing object-column value as NaN, and NaN is
# TRUTHY, so `if k` let it through and sorted() then compared a float to a str. (2026-09-04)
_kinds = sorted({k for k in M["kind"] if isinstance(k, str) and k})
_unclassified = [
    (c["label"], (c.get("actual_path") or lc.full_path(c["api"], c["path"]) or "").split("?")[0])
    for c, k in zip(ALL_CALLS, M["kind"]) if not (isinstance(k, str) and k)
]
print(f"  seen: {_kinds}")
print(f"  missing: {sorted(set(lc.RESOURCE_KINDS) - set(_kinds)) or 'none -- all six driven'}")
if _unclassified:
    # classify_paths drives each path through its own window and reads back the resource row
    # it created. It returns (None, None) when that window holds no row with a non-zero count
    # -- which means the synthetic line did not parse, so the real one would land in
    # `parse_errors` rather than on any resource. Worth seeing, not worth failing on.
    print(f"  UNCLASSIFIED: {len(_unclassified)} of {len(M)} calls -- the deployed classify() "
          "produced no resource row for these paths")
    for label, path in _unclassified[:12]:
        print(f"    {label[:36]:36} {path[:70]}")

print("\n--- discrepancies (expected vs actual) ---")
_d = M[M.discrepancy != ""]
print("none" if _d.empty else _d.to_string(index=False))

print("\n--- probe expectations (predicted across the WHOLE run, in issue order) ---")
EXP_BY_RID = {c["request_id"]: (e or {}).get("verdict") for c, e in zip(ALL_CALLS, EXPECTED)}
for p in PROBES:
    want = sum(1 for row in p["rows"] if EXP_BY_RID.get(row["request_id"]) == "keep")
    got = (sum(len([r for r in BY_RID.get(row["request_id"], []) if vlogs.is_access_log(r)])
               for row in p["rows"]) if CORRELATION == "exact" else None)
    verdict = "" if got is None or got == want else "   <-- MISMATCH"
    print(f"  {p['probe']:44} calls {p['calls']:<3} expected {want:<3} got {got}{verdict}")

print("\n--- WARN / ERROR messages seen (source plan section 8.1) ---")
WARNS = V.query(vlogs.and_(vlogs.app_polaris(), 'level:"WARN" OR level:"ERROR"'),
                start=WINDOW_START, end=WINDOW_END, limit=1000)
DISTINCT_WARNS = {}
for w in WARNS:
    key = (w.get("level"), (w.get("_msg") or "")[:120])
    DISTINCT_WARNS[key] = DISTINCT_WARNS.get(key, 0) + 1
for (lvl, msg), n in sorted(DISTINCT_WARNS.items(), key=lambda kv: -kv[1]):
    print(f"  {lvl:5} x{n:<4} {msg}")
if not DISTINCT_WARNS:
    print("  none -- so the 'Deprecated Config' exclusion still has no message to match on.")

print("\n--- did any record carry an exception / stack trace? ---")
# NAMES, NOT A BOOLEAN. `"exception" in r or "stackTrace" in str(r)` reported 0 of 5
# on 2026-09-07 and a `grep -c stackTrace` on the source log returned 0, and the two
# were read as corroboration. They were the same error twice: this build emits the
# STRUCTURED exception output -- an object with a `frames` array of {class, method,
# line} -- so there is no `stackTrace` string, and a flattened `exception.frames`
# defeats a top-level key test. See lc.exception_fields.
EXC = [r for r in WARNS if lc.exception_fields(r)]
EXC_FIELDS = sorted({f for r in WARNS for f in lc.exception_fields(r)})
print(f"  {len(EXC)} of {len(WARNS)} WARN/ERROR records carry an exception payload")
print(f"  fields carrying it: {EXC_FIELDS or 'none'}")
if EXC:
    _f = lc.exception_fields(EXC[0])[0].split(" (")[0]
    print(f"  -> {str(EXC[0].get(_f))[:300]}")
else:
    print("  -> NOT DEMONSTRATED, and the FIRST thing to check is the field name, not the")
    print("     pipeline. List the keys of an ERROR record at the source before concluding:")
    print("     kubectl -n datahub-hynix exec deploy/benchmarks-polaris -- sh -c \\")
    print("       \"grep -m1 INTERNAL_SERVER_ERROR /deployments/logs/polaris.log\" \\")
    print("       | python3 -c \"import json,sys; print(sorted(json.load(sys.stdin)))\"")


  resource_row is the key the path classifies to. For the 40 error calls it is where the call WOULD have landed:
  an error on a resource not already known increments __other__ instead.
                                                          label method                                                       path  status   ms expected disposition  stored  app_lines                                                                                      resource_row       kind           principal_row discrepancy
                                             iceberg.get_config    GET  /api/catalog/v1/config?warehouse=apiprofile1788745242_cat   200.0  6.1     drop     counted       0         10                                                                            /api/catalog/v1/config      other nb_1788745242_principal            
                                        iceberg.list_namespaces    GET        /api/catalog/v1/apiprofile1788745242_cat/namespaces   200.0  5.7     drop     c

## 9 — Cleanup

The cleanup DELETEs are part of the test — they are driven inside the tagged window, not after it. A failed run must still leave the cluster usable.

In [11]:
CLEANUP = {}
CLEANUP["fixture"] = surf.teardown_fixture(FX, adm_pc, adm_ic, sweep_stale=False)
CLEANUP["run_principal"] = lc.deprovision_run_principal(adm_pc, NB_PRINCIPAL, NB_PROLE)
CLEANUP["denied_principal"] = lc.deprovision_run_principal(adm_pc, DENIED, DENIED_ROLE)
try:
    adm_pc.delete_catalog(f"nb{RUN}npe", purge=True)
except Exception:
    pass

for k, v in CLEANUP.items():
    print(f"  {k:20} {v}")

try:
    del pull
except NameError:
    pass
import gc
gc.collect()
print("\n  cleaned. Anything left behind is listed above.")

  fixture              {'fixture': {'tables': 1, 'views': 0, 'namespaces': 2, 'catalog_roles': 1, 'catalog': 204, 'failed': {}, 'builtin_role_skipped': 'catalog_admin'}, 'swept': {}}
  run_principal        []
  denied_principal     []

  cleaned. Anything left behind is listed above.


## 10 — The scheduled report, across every window the run touched

**A run does not fit in one window, and at `WINDOW_SECONDS: 30` it never will.** The first run
of this notebook read a single window and found 8 records, all errors, one `__other__` row — the
cleanup DELETEs and nothing else, because that is what happened to land in the last 30 seconds.
It looked exactly like the pipeline losing the entire run.

So this cell collects **every window from the one containing `STARTED` to the one containing the
last call**, checks the schema invariants on each window individually — that is where they have
to hold — and merges them into one comparable set for cell 12.

It then captures the two windows *after* the run, in the same cell and in a tight loop.
Zero-carry only survives one window, so a user-paced gap between cells is enough to lose it:
that is what void'd the first run's decay check.

In [12]:
REPORT, LAG = {}, max(8.0, INGEST_LAG_S * 3)
RUN_END = datetime.now(timezone.utc)
_first = lc.window_bounds(STARTED.timestamp(), WINDOW_SECONDS)[0]
_last = lc.window_bounds(RUN_END.timestamp(), WINDOW_SECONDS)[0]
_n = (_last - _first) // WINDOW_SECONDS + 1
print(f"  the run spans {_n} window(s) of {WINDOW_SECONDS}s: "
      f"{lc._iso_z(_first)} .. {lc._iso_z(_last + WINDOW_SECONDS)}")

_due = _last + WINDOW_SECONDS + LAG
if time.time() < _due:
    print(f"  waiting {_due - time.time():.0f}s for the last one to flush")
    time.sleep(max(0, _due - time.time()))

RUN_WINDOWS = [lc._iso_z(_first + i * WINDOW_SECONDS) for i in range(_n)]
BY_WINDOW = {ws: V.report_window(ws, since="6h") for ws in RUN_WINDOWS}
_empty = [ws for ws, rows in BY_WINDOW.items() if not rows]
print(f"  windows with a report: {_n - len(_empty)}/{_n}"
      + (f"   MISSING: {_empty}" if _empty else ""))
if _empty:
    print("   -> a window with no report at all is a SKIPPED WINDOW (a late tick moved the")
    print("      filter's index by two) or a shipper restart. Cell 14 tells them apart.")

# ONE LINE PER WINDOW. `merged rows: 49` on its own is unreadable, and run 1 printed
# exactly that beside "windows the run spanned: 1" -- a single window holding 49 rows
# for a run with 30 distinct resource keys. Whether a run was split, and how its traffic
# fell across the boundaries, has to be visible without re-querying.
for _ws in RUN_WINDOWS:
    _rows = BY_WINDOW[_ws]
    _s = next((r for r in _rows if r.get("report_type") == "summary"), {})
    print(f"   {_ws}  rows={len(_rows):>3}  access_seen={str(_s.get('access_seen', '-')):>4}"
          f"  kept={str(_s.get('access_kept', '-')):>4}"
          f"  resources={str(_s.get('distinct_resources', '-')):>3}"
          f"  partial={_s.get('partial_window', '-')}")

# The schema has to hold per window; the merge is only for comparing against the oracle.
PER_WINDOW_BAD = {ws: lc.check_invariants(rows) for ws, rows in BY_WINDOW.items() if rows}
_bad_windows = {ws: b for ws, b in PER_WINDOW_BAD.items() if b}
print(f"  per-window invariants: {'all OK' if not _bad_windows else 'FAILURES'}")
for ws, b in list(_bad_windows.items())[:5]:
    print(f"   !! {ws}: {b}")

W1 = lc.merge_windows(BY_WINDOW.values())
BAD = lc.check_invariants(W1, merged=True)
print(f"  merged invariants: {'OK' if not BAD else ''}")
for b in BAD:
    print("   !!", b)

if not W1:
    raise RuntimeError(
        "no report rows at all for the run's windows. Either the shipper is not running, "
        "or the report OUTPUT is not wired to VictoriaLogs -- cell 0b would have caught both, "
        "so check whether the shipper restarted mid-run.")

_split = {}
for r in W1:
    _split[r["report_type"]] = _split.get(r["report_type"], 0) + 1
if sorted(_split) != ["principal", "resource", "summary"]:
    raise RuntimeError(
        f"the array return did not split into three report_types: {_split}. One record "
        "carrying numeric-keyed fields means this schema does not exist; the fix is in "
        "local-k8s/logging/fb-values.yaml, not here.")

S1 = [r for r in W1 if r["report_type"] == "summary"][0]
RES1 = [r for r in W1 if r["report_type"] == "resource"]
PRI1 = [r for r in W1 if r["report_type"] == "principal"]
print(f"\n  MERGED {lc._iso_z(_first)} .. {S1['window_end']}   host={S1['hostname']}"
      f"  partial={S1['partial_window']!r}")
print(f"  seen={S1['access_seen']}  kept={S1['access_kept']}  counted={S1['access_counted']}"
      f"  (GET {S1['counted_get']}, POST {S1['counted_post']})  errors_kept={S1['errors_kept']}"
      f"  parse_errors={S1['parse_errors']}")
print(f"  distinct: {len(RES1)} resources, {len(PRI1)} principals"
      f"   __other__ hits: {S1['resources_other']}")
print(f"  calls driven: {len([c for c in ALL_CALLS if c.get('status')])}"
      "   (the gap is the token exchanges each client makes on construction --")
print("    they are real access lines and are attributed to principal '-')")

print("\n  --- resource rows (merged) ---")
for r in sorted(RES1, key=lambda r: -int(r.get("requests", 0) or 0))[:25]:
    print(f"   {r['resource'][:60]:60} {r['resource_kind']:11} req={r['requests']:>4}"
          f" rd={r['reads']:>4} wr={r['writes']:>4} err={r['errors']:>4}"
          f" bytes={r['response_bytes']:>8}")
print("  --- principal rows (merged) ---")
for r in sorted(PRI1, key=lambda r: -int(r.get("requests", 0) or 0)):
    print(f"   {str(r['user_principal_name'])[:40]:40} req={r['requests']:>4}"
          f" rd={r['reads']:>4} wr={r['writes']:>4} err={r['errors']:>4}")

KINDS_SEEN = sorted({r["resource_kind"] for r in RES1})
print(f"\n  resource_kind seen: {KINDS_SEEN}")
print(f"  missing           : {sorted(set(lc.RESOURCE_KINDS) - set(KINDS_SEEN)) or 'none'}")
print(f"  no /metrics row   : {not any('/metrics' in r['resource'] for r in RES1)}")

# A REPLAY LOOKS LIKE A SPIKE: emptyDir tail DB + Read_from_Head means a `helm upgrade`
# re-reads the whole file, and VictoriaLogs does not deduplicate on ingest.
_span = None
if S1.get("min_record_time") and S1.get("max_record_time"):
    _span = (lc._epoch_of(S1["max_record_time"][:19] + "Z")
             - lc._epoch_of(S1["min_record_time"][:19] + "Z"))
_merged_span = _n * WINDOW_SECONDS
print(f"  record span: {_span}s across a {_merged_span}s merged range")
if _span is not None and _span > _merged_span:
    print("   !! WIDER THAN THE RANGE -- this is a REPLAY, not traffic. Every error and")
    print("      mutation replays as a full record. Do not read these counts as a spike.")

REPORT["run"] = {"windows": RUN_WINDOWS, "by_window": BY_WINDOW, "merged": W1,
                 "last": lc._iso_z(_last), "end": lc._iso_z(_last + WINDOW_SECONDS),
                 "invariants": BAD, "kinds": KINDS_SEEN, "per_window": PER_WINDOW_BAD,
                 "rows_by_window": {ws: len(rows) for ws, rows in BY_WINDOW.items()},
                 "started": STARTED.timestamp(), "ended": RUN_END.timestamp()}

# ---- the carry sequence, captured here so no user-paced gap can lose a window ----
# Zero-carry lives for exactly ONE window. On the first run the decay cell ran 53s after the
# window it wanted had already closed, and correctly declared itself void.
print(f"\n  --- capturing the two windows after the run (tight loop, ~{2 * WINDOW_SECONDS}s) ---")
QUIET = []
for _i in (1, 2):
    _s0 = _last + _i * WINDOW_SECONDS
    _d = _s0 + WINDOW_SECONDS + LAG
    if time.time() < _d:
        print(f"   waiting {_d - time.time():.0f}s for {lc._iso_z(_s0)}")
        time.sleep(max(0, _d - time.time()))
    QUIET.append((lc._iso_z(_s0), V.report_window(lc._iso_z(_s0), since="6h")))
    print(f"   {lc._iso_z(_s0)}: {len(QUIET[-1][1])} rows")
REPORT["quiet"] = QUIET


  the run spans 1 window(s) of 30s: 2026-09-07T01:40:30Z .. 2026-09-07T01:41:00Z
  waiting 11s for the last one to flush
  windows with a report: 1/1
   2026-09-07T01:40:30Z  rows= 48  access_seen= 250  kept= 111  resources= 43  partial=false
  per-window invariants: all OK
  merged invariants: OK

  MERGED 2026-09-07T01:40:30Z .. 2026-09-07T01:41:00Z   host=fb-polaris-shipper-fluent-bit-68b4959db4-4f7tf  partial='false'
  seen=250  kept=111  counted=139  (GET 115, POST 24)  errors_kept=49  parse_errors=0
  distinct: 43 resources, 4 principals   __other__ hits: 31
  calls driven: 145   (the gap is the token exchanges each client makes on construction --
    they are real access lines and are attributed to principal '-')

  --- resource rows (merged) ---
   /api/management/v1/catalogs/apiprofile1788745242_cat/catalog management  req=  50 rd=  25 wr=  25 err=   0 bytes=   16205
   /api/catalog/v1/apiprofile1788745242_cat/namespaces/probe_ns table       req=  48 rd=  38 wr=  10 err=  10 b

## 11 — The oracle diff

The run's own calls are replayed through the deployed Lua with a tick on the boundary, and the
array that comes back **is** the expected report. Diffing it against the merged windows makes
schema coverage a diff rather than a set of hand-written expectations.

Excluded, each for a stated reason: `hostname` and `report_seq` describe the emitting *process*;
`window_start` / `window_end` / `_time` / `partial_window` describe *which* window, and the
oracle's synthetic one is not the real one; `response_bytes` is not predictable from a client
that has no `%b`; `min` / `max_record_time` come from the real records' own timestamps.

Each call is replayed as **the principal that issued it**, read from its own `Authorization`
header — the management block drives as root and the 403 batch as `nb_<run>_denied`, so
replaying everything as one identity would put a disagreement into every principal row and
leave it to be excused in prose.

**The window holds more than this run.** Every client construction exchanges an OAuth token, and
those lines are real traffic attributed to principal `-`. So the summary counters are read as a
*delta to explain*, and strict equality is asserted only on the fixture's own resource rows.

In [13]:
DIFF_IGNORE = lc.VOLATILE_FIELDS | lc.VLOGS_META | {
    # `_stream` / `_stream_id` are fields VictoriaLogs ADDS on the way out -- the
    # filter never emitted them, so the oracle cannot have them and every stored row
    # was contributing two guaranteed mismatches. 34 of run 1788744260's 60 listed
    # fixture mismatches were these. `check_invariants` learned it on 2026-09-04
    # (VLOGS_META) and this cell never did.
    "response_bytes",                       # no %b client-side; see PLAN section 2.5
    "min_record_time", "max_record_time",   # from the real records' own timestamps
    "window_start", "window_end", "_time",  # which window, not what it counted
    "partial_window",                       # the oracle's first window is always partial
}

_synth = []
for c in ALL_CALLS:
    if not c.get("status"):
        continue          # a call that never completed leaves no access-log line at all
    # AS THE PRINCIPAL THAT ISSUED IT. Replaying every call as one identity makes
    # the oracle's principal rows disagree with the pipeline's by construction, and
    # the disagreement then has to be excused in prose rather than read. The
    # management block goes out as root and the 403 batch as nb_<run>_denied.
    _synth.append(lc.access_log_record(
        c["method"],
        c.get("actual_path") or lc.full_path(c["api"], c["path"]),
        c["status"],
        user=c.get("principal") or DRIVE_PRINCIPAL,
    ))

_, _oracle = POLICY.report_windows(_synth, silent_windows=0)
EXPECTED_ROWS = _oracle["w1"]
print(f"  oracle: {len(EXPECTED_ROWS)} rows from {len(_synth)} replayed calls")
print("  oracle invariants:", lc.check_invariants(EXPECTED_ROWS) or "OK")

DIFF = lc.diff_reports(EXPECTED_ROWS, W1, ignore=DIFF_IGNORE)
MIS = lc.report_mismatches(DIFF)
print(f"  diff: {len(DIFF) - len(MIS)} matching fields, {len(MIS)} mismatches\n")

import pandas as pd
_sum_mis = [m for m in MIS if m["report_type"] == "summary"]
if _sum_mis:
    print("  --- summary counters: the delta to explain ---")
    for m in _sum_mis:
        print(f"   {m['field']:22} oracle={str(m['expected'])[:24]:24} stored={m['actual']}")
    print("   (stored > oracle is expected: token exchanges are not in ALL_CALLS)")

_row_mis = [m for m in MIS if m["report_type"] != "summary"]
if _row_mis:
    D = pd.DataFrame(_row_mis)
    D["key"] = D["key"].astype(str).str.slice(-52)
    print(f"\n  --- row mismatches ({len(_row_mis)}) ---")
    print(D.to_string(index=False, max_colwidth=54))

# STRICT on the fixture's own resources: those paths are this run's alone, so a mismatch there
# is the pipeline rather than the neighbours.
OURS = [m for m in MIS if FX.cat in str(m["key"])]
print(f"\n  mismatches on THIS RUN'S fixture resources: {len(OURS)}"
      + ("   <-- THE ASSERTION THAT MATTERS IS FAILING" if OURS else "   OK"))
if OURS:
    print("     These paths are this run's alone, so a mismatch here is the pipeline"
          " disagreeing\n     with the deployed Lua -- not a neighbour's traffic. Run 1"
          " reported 66 of these\n     as a bare number in the results document and named"
          " none of them.")
for m in OURS[:20]:
    print(f"   {m['report_type']:9} {str(m['key'])[-52:]:52} {m['field']:16}"
          f" expected={m['expected']!r} actual={m['actual']!r}")

_tbl_key = f"/api/catalog/v1/{FX.cat}/namespaces/{FX.ns}/tables/{FX.tbl}"
_row = next((r for r in RES1 if r["resource"] == _tbl_key), None)
if _row is None:
    print(f"\n  !! no resource row for {_tbl_key}")
    print("     Under v3 a successful read is COUNTED, so the row exists only if the table was")
    print("     touched inside one of the run's windows -- check the merged range covers cell 3.")
else:
    _nz = {k: _row[k] for k in ("requests", "reads", "writes", "errors")}
    print(f"\n  fixture table row: {_nz}")
    print("  all four non-zero:", all(int(v) > 0 for v in _nz.values()),
          "  (errors=0 is EXPECTED if the 403 client could not be provisioned)")
REPORT["diff"] = {"mismatches": MIS, "ours": OURS}


  oracle: 35 rows from 145 replayed calls
  oracle invariants: OK
  diff: 326 matching fields, 74 mismatches

  --- summary counters: the delta to explain ---
   access_counted         oracle=89                       stored=139
   access_kept            oracle=56                       stored=111
   access_seen            oracle=145                      stored=250
   counted_get            oracle=69                       stored=115
   counted_post           oracle=20                       stored=24
   distinct_principals    oracle=3                        stored=4
   distinct_resources     oracle=31                       stored=43
   errors_kept            oracle=40                       stored=49
   resources_other        oracle=25                       stored=31
   (stored > oracle is expected: token exchanges are not in ALL_CALLS)

  --- row mismatches (65) ---
report_type                                                  key    field expected        actual  status
  principal        

## 11b — the matrix, completed: `window_start`, `counted_where`, and the arithmetic

PLAN section 7 names five columns and run 1 shipped three. The two missing ones are the ones
that let a reader check the report rather than trust it: `window_start` says which window a
call was counted into, and `counted_where` names the row that actually moved — which is what
turns a count of mismatches into a mismatch.

Both need cell 10's report, so they are added here rather than in cell 8. This cell also does
the three reconciliations run 1 left to the reader: the merged row count against the rows this
run can account for, the assertions PLAN section 7 names, and the per-principal mix that makes
the margin equality something other than a tautology.


In [14]:
# The window a call was counted into. A counted call leaves no individual record, so
# `issued_at` (recorded at the call, cell 8 and the drivers) is the only source there is.
M["window_start"] = [
    lc._iso_z(lc.window_bounds(c["issued_at"], WINDOW_SECONDS)[0])
    if c.get("issued_at") else ""
    for c in ALL_CALLS
]
# The row that ACTUALLY moved -- not the key the path classifies to. An error on a
# resource nobody read successfully increments __other__ instead, so for a 4xx row
# `resource_row` is a counterfactual and this column is the observation.
M["counted_where"] = lc.counted_where(ALL_CALLS, KEYS, W1)

print("  calls per report window:")
for _ws, _n in sorted(M[M.window_start != ""].window_start.value_counts().items()):
    print(f"   {_ws}  {_n:>4} calls")
_absent = M[M.counted_where == "ABSENT"]
print(f"  calls whose row is absent from the merged report: {len(_absent)}")
if len(_absent):
    print(_absent[["label", "status", "resource_row"]].head(12).to_string(index=False))

# --- does the merged report hold the rows this run can account for? ---
RECON = lc.reconcile_merged_rows(W1, ALL_CALLS, KEYS)
_cnt = RECON["counts"]
print(f"\n  merged rows {_cnt['total']} = 1 summary + {_cnt['resource']} resources"
      f" + {_cnt['principal']} principals")
print(f"  resource keys this run explains: {_cnt['resource_expected']};"
      f"  unexplained: {len(RECON['unexplained'])};  missing: {len(RECON['missing'])}")
for _k in RECON["unexplained"][:15]:
    print(f"   unexplained  {str(_k)[-70:]}")
for _k in RECON["missing"][:15]:
    print(f"   MISSING      {str(_k)[-70:]}")
if RECON["unexplained"]:
    print("   -> a row this run cannot account for is a carry from a window that opened")
    print("      before STARTED, or another client on the cluster. Not automatically wrong")
    print("      -- but an unreconciled row count is what made run 1's 49 unreadable.")

# --- the assertions PLAN section 7 names, stated rather than implied ---
# No wall-clock bounds: the record times belong to the WINDOWS, and the merged range
# runs past RUN_END and opens before STARTED by construction. See lc.named_assertions.
ASSERTS = lc.named_assertions(W1, S1)
print("\n  --- named assertions ---")
for _name, _ok, _detail in ASSERTS:
    print(f"   {'PASS' if _ok else ('n/a ' if _ok is None else 'FAIL')}  {_name:58} {_detail}")

# --- the per-principal margin, which one principal cannot test ---
MIX = lc.principal_mix(W1)
print("\n  --- principal rows (the margin's only real content) ---")
for _p, _v in sorted(MIX.items(), key=lambda kv: -kv[1]["requests"]):
    print(f"   {str(_p)[:40]:40} {_v}")
_tot = sum(_v["requests"] for _v in MIX.values())
_net = lc._as_int(S1.get("access_seen")) - lc._as_int(S1.get("parse_errors"))
print(f"  sum(principal.requests)={_tot}   access_seen-parse_errors={_net}"
      f"   {'OK' if _tot == _net else '<-- MARGIN BROKEN'}")
_strict = [_p for _p, _v in MIX.items() if 0 < _v["requests"] < _tot]
print(f"  principals carrying a strict fraction of the total: {len(_strict)}"
      + ("   OK -- the equality is not satisfied by a global counter"
         if len(_strict) >= 2 else
         "   <-- ONE identity: the margin is tautological this run (PLAN 6.2)"))

REPORT["recon"], REPORT["asserts"], REPORT["mix"] = RECON, ASSERTS, MIX
REPORT["mgmt"], REPORT["volume"] = MGMT_POSTS, VOLUME


  calls per report window:
   2026-09-07T01:40:30Z   146 calls
  calls whose row is absent from the merged report: 0

  merged rows 48 = 1 summary + 43 resources + 4 principals
  resource keys this run explains: 31;  unexplained: 12;  missing: 0
   unexplained  /api/catalog/v1/apiprofile1788745242_cat/namespaces/probe_ns2
   unexplained  /api/catalog/v1/apiprofile1788745242_cat/namespaces/probe_ns2/tables
   unexplained  /api/catalog/v1/apiprofile1788745242_cat/namespaces/probe_ns2/views
   unexplained  ogs/apiprofile1788745242_cat/catalog-roles/apiprofile1788745242_shared
   unexplained  profile1788745242_cat/catalog-roles/apiprofile1788745242_shared/grants
   unexplained  /api/management/v1/principal-roles/nb_1788745242_denied_role
   unexplained  /api/management/v1/principal-roles/nb_1788745242_prole
   unexplained  cipal-roles/nb_1788745242_prole/catalog-roles/apiprofile1788745242_cat
   unexplained  /api/management/v1/principals/nb_1788745242_denied
   unexplained  /api/management

## 12 — Zero-carry: a resource that goes quiet still reports

Captured in cell 11; nothing is waited for here. A resource active in the run's **last** window
must appear in the next one with an explicit `0` — carry is window-to-window, so the comparison
is against that last window and not against the merged range.

Zero-carry is what lets a dashboard tell *no traffic* from *no shipper*. Without it a series
just disappears, and disappearing is what an outage looks like too.

In [15]:
LAST_WS = REPORT["run"]["last"]
LAST_ROWS = REPORT["run"]["by_window"].get(LAST_WS, [])
W2_START, W2 = REPORT["quiet"][0]
res_last = {r["resource"]: r for r in LAST_ROWS if r["report_type"] == "resource"}
ACTIVE = {k for k, r in res_last.items() if int(r.get("requests", 0) or 0) > 0}
res2 = {r["resource"]: r for r in W2 if r["report_type"] == "resource"}
S_last = [r for r in LAST_ROWS if r["report_type"] == "summary"]
S2 = [r for r in W2 if r["report_type"] == "summary"]

print(f"  run's last window {LAST_WS}: {len(ACTIVE)} active resources")
print(f"  next window {W2_START}: {len(W2)} rows")
print("  invariants:", lc.check_invariants(W2) or "OK")

if not W2:
    print("   !! NO REPORT for this window -- a skipped window (late tick) or a restart.")
elif not ACTIVE:
    print("   !! the run's last window saw no traffic, so there is nothing to carry. VOID,")
    print("      not failed -- the run ended too far before the boundary.")
else:
    carried = ACTIVE & set(res2)
    zeroed = {k for k in carried if int(res2[k].get("requests", 0) or 0) == 0}
    print(f"  ZERO-CARRY: {len(carried)}/{len(ACTIVE)} carried"
          + ("" if carried == ACTIVE else f"   MISSING {sorted(ACTIVE - carried)[:3]}"))
    print(f"              {len(zeroed)}/{len(carried)} report an explicit 0")
    if S_last and S2:
        _a, _b = int(S_last[0]["report_seq"]), int(S2[0]["report_seq"])
        print(f"  report_seq  {_a} -> {_b}   {'OK' if _b == _a + 1 else '<-- NOT +1'}")
        print(f"  same host   {S2[0]['hostname'] == S_last[0]['hostname']}"
              + ("" if S2[0]["hostname"] == S_last[0]["hostname"]
                 else "   <-- THE SHIPPER RESTARTED: the counts are void"))
        print(f"  counters reset: access_seen={S2[0]['access_seen']}"
              f" (last run window saw {S_last[0]['access_seen']})")
REPORT["w2"] = {"window": W2_START, "rows": W2,
                "carried": sorted(ACTIVE & set(res2)) if W2 else []}


  run's last window 2026-09-07T01:40:30Z: 43 active resources
  next window 2026-09-07T01:41:00Z: 48 rows
  invariants: OK
  ZERO-CARRY: 43/43 carried
              43/43 report an explicit 0
  report_seq  225 -> 226   OK
  same host   True
  counters reset: access_seen=0 (last run window saw 250)


## 13 — Carry decay, and the tick is not the period

The other half: a row that stayed at `0` is **not** carried a third time. Without decay every
key ever seen is reported forever.

And the property the whole design turns on, end to end at last: the dummy INPUT ticks every
`Interval_Sec`, so a window holds several ticks. If the filter emitted on the tick rather than
on the boundary there would be several reports per window. Count them.

A **gap** between consecutive `window_start`s means a tick arrived late enough that the window
index jumped and the window in between was never reported. That is only meaningful within one
process at one window length — the first run flagged the `1800 → 30` change itself.

In [16]:
W3_START, W3 = REPORT["quiet"][1]
res3 = {r["resource"] for r in W3 if r["report_type"] == "resource"}
print(f"  window {W3_START}: {len(W3)} rows")
print("  invariants:", lc.check_invariants(W3) or "OK")

_stale = set(REPORT["w2"]["carried"])
_still = _stale & res3
print(f"\n  CARRY DECAY: {len(_stale)} rows sat at 0 through the previous window")
print(f"               {len(_still)} still reported"
      + ("   OK -- decayed" if not _still else f"   <-- CARRIED AGAIN: {sorted(_still)[:3]}"))

_all = V.reports(report_type="summary", since="6h", limit=5000)
_by = {}
for r in _all:
    _k = (r.get("hostname"), r.get("window_start"))
    _by[_k] = _by.get(_k, 0) + 1
_ticks = WINDOW_SECONDS / (TICK_SECONDS or 30)
print(f"\n  TICK != PERIOD: the dummy INPUT ticks every {TICK_SECONDS}s, so ~{_ticks:.0f} ticks"
      f" fall inside each {WINDOW_SECONDS}s window")
print(f"  summaries per (host, window): {sorted(set(_by.values()))}"
      f"   {'OK -- exactly one' if set(_by.values()) == {1} else '<-- ' + str({k: v for k, v in _by.items() if v > 1})[:180]}")

# A GAP IS ONLY A SKIPPED WINDOW WITHIN ONE PROCESS AT ONE SETTING. The 2026-09-04 verify run
# flagged 07:30:00Z -> 08:14:00Z, which was the WINDOW_SECONDS change from 1800 to 30 and the
# shipper restart that carried it -- not a late tick.
_mine = [r for r in _all
         if r.get("hostname") == S1["hostname"]
         and lc._as_int(r.get("window_seconds")) == WINDOW_SECONDS]
_starts = sorted({lc._epoch_of(r["window_start"]) for r in _mine if r.get("window_start")})
_gaps = [(a, b) for a, b in zip(_starts, _starts[1:]) if b - a != WINDOW_SECONDS]
print(f"  consecutive windows from this shipper at {WINDOW_SECONDS}s: {len(_starts)}"
      f"   ({len(_all) - len(_mine)} rows from another process or window length, excluded)")
print(f"  gaps: {'none' if not _gaps else [(lc._iso_z(a), lc._iso_z(b)) for a, b in _gaps]}")
if _gaps:
    print(f"   -> within ONE process at {WINDOW_SECONDS}s with a {TICK_SECONDS}s tick "
          f"({_ticks:.0f} ticks per window) a gap is a FINDING, not the setup.")

print(f"\n  raw tick leak (must be 0): {V.count(vlogs.tick_leak(), start='6h')}")
REPORT["w3"] = {"window": W3_START, "rows": W3, "decayed": not _still, "gaps": _gaps}


  window 2026-09-07T01:41:30Z: 1 rows
  invariants: OK

  CARRY DECAY: 43 rows sat at 0 through the previous window
               0 still reported   OK -- decayed

  TICK != PERIOD: the dummy INPUT ticks every 5s, so ~6 ticks fall inside each 30s window
  summaries per (host, window): [1]   OK -- exactly one
  consecutive windows from this shipper at 30s: 131   (0 rows from another process or window length, excluded)
  gaps: [('2026-09-07T00:35:30Z', '2026-09-07T00:37:30Z')]
   -> within ONE process at 30s with a 5s tick (6 ticks per window) a gap is a FINDING, not the setup.

  raw tick leak (must be 0): 0


## 14 — Findings


In [17]:
def q(x):
    return "unknown" if x is None else x


# WHICH LOGGER DOMINATES? The filter governs only the access-log records; every application
# line rides through untouched by rule 2. The previous run left "which logger dominates" open,
# and it is the input to the only remaining volume decision.
LOGGERS = {}
for rec in STORED:
    _l = rec.get("loggerName", "(none)")
    LOGGERS[_l] = LOGGERS.get(_l, 0) + 1
ACCESS_STORED = sum(1 for r in STORED if vlogs.is_access_log(r))
APP_STORED = len(STORED) - ACCESS_STORED
print("  --- stored records by loggerName ---")
for _l, _n in sorted(LOGGERS.items(), key=lambda kv: -kv[1])[:12]:
    print(f"   {_n:>6}  {_l}")
print(f"  access-log records: {ACCESS_STORED} of {len(STORED)} "
      f"({100 * ACCESS_STORED / max(1, len(STORED)):.1f}%) -- the filter governs only these")

#: A 500 that did not happen is not evidence that stack traces are dropped. `0 of 0` has to
#: say so out loud, or the next reader quotes it as a result.
_npe = [c for c in ALL_CALLS if "500" in c["label"]]
Q3_NOTE = "" if WARNS else (
    "\n\n   **NOT ANSWERED, and the reason is a probe that stopped provoking its error"
    " rather than a property of the pipeline.** "
    + (f"`{_npe[0]['label']}` returned **{_npe[0]['status']}** this run, so no 500 was "
       "provoked and no ERROR record exists to inspect. " if _npe else
       "No call in this run produced a 500. ")
    + "`0 of 0` is not evidence that stack traces are dropped. Fix the probe, or drive a "
      "known 500 from `error-cases/`, before quoting anything here."
)

# THE TOP OF THE ORACLE DIFF, so the report is readable by someone who was not here. A count
# of mismatches is not a finding; the fields that moved are.
_top = REPORT["diff"]["mismatches"][:20]

report = f"""# Log coverage — run {RUN}

Generated by `polaris_log_coverage.ipynb` on {STARTED.date().isoformat()}.
Correlation mode **{CORRELATION}**; drove as **{DRIVE_PRINCIPAL}**.
Expected column computed from `{POLICY.source if POLICY else 'UNAVAILABLE'}`
(sha256 `{POLICY.sha256[:16] if POLICY else '-'}`).

## Validity

| | |
|---|---|
| shipper pod before | `{SHIPPER_BEFORE}` |
| shipper pod after | `{q(SHIPPER_AFTER)}` |
| Polaris replicas before / after | `{HPA_BEFORE}` / `{q(HPA_AFTER)}` |
| duplicate records (shipper replay) | {DUPES} |
| ingest lag, first record | {INGEST_LAG_S:.1f}s |
| numeric `http_status` filters, instrument check | {q(NUMERIC_DETAIL)} |
| numeric `http_status` filters, THIS run | {q(NUMERIC_RUN_DETAIL)} |
| records lost in transit (output, since pod start) | {"none — errors/retries_failed/dropped all 0" if OUT_HEALTH and not OUT_HEALTH["lost"] else (OUT_HEALTH or FB_OUT_ERR)} |
| retention policy deployed? | {q(POLICY_LIVE)} — {POLICY_LIVE_DETAIL.splitlines()[0]} |
| records per call | {len(STORED) / max(1, len(ALL_CALLS)):.1f} |

{"**THIS RUN IS INVALID** — Polaris scaled mid-run." if HPA_MOVED else ""}
{"**Dedup expectations are VOID** — the shipper restarted mid-run." if SHIPPER_MOVED else ""}

## The seven questions

1. **Distinct WARN messages**{f" — {len(DISTINCT_WARNS)} seen:" if DISTINCT_WARNS else ""}
{chr(10).join(f"   - `{lvl}` x{n}: {msg}" for (lvl, msg), n in DISTINCT_WARNS.items()) or "   - **NOT ANSWERED.** The run produced no WARN or ERROR record at all, so there is nothing to write the Deprecated Config exclusion against. Rule 1 was not exercised. See question 3."}
2. **Where a PUT's body is logged** — {len(priv_hits)} record(s) in VictoriaLogs carry the
   granted privilege name. eventListener `events` table: {"%s rows in schema %s" % (EVENTS["rows"], EVENTS["schema"]) if EVENTS["reachable"] else EVENTS["note"]}.
3. **Do stack traces survive?** — {len(EXC)} of {len(WARNS)} WARN/ERROR records carry an
   exception payload{", in " + ", ".join(f"`{f}`" for f in EXC_FIELDS) if EXC_FIELDS else ""}.
   {"" if not EXC_FIELDS else "The field NAME is reported because absence and a mis-named search are indistinguishable otherwise, and they have opposite remedies: on 2026-09-07 this line read `0 of 5` and a `grep -c stackTrace` agreed with it, and both were looking for a name this build does not use — the traces were structured, under a `frames` array, all along."}{Q3_NOTE}
4. **Does `mdc.requestId` join reliably?** — {CORRELATION}; **{CORR["recovered"]} of
   {CORR["with_id"]}** calls that carried an id had it recovered from VictoriaLogs
   ({len(ALL_CALLS)} calls driven; {CORR["other_ids"]} further run-minted id(s) in the pull
   belong to calls outside this table, such as cell 1's correlation probe).{
   "" if not CORR["missing"] else chr(10) + "   **NOT RECOVERED:** " + ", ".join(f"`{m}`" for m in CORR["missing"][:8])}
5. **The v2 audit hole, and whether v3 closed it** — {int((M.expected == "drop").sum())} of {len(M)}
   calls leave no individual record; every one of them is COUNTED into the window aggregate.
   Management POSTs kept: **{REPORT["mgmt"]["kept"]} of {REPORT["mgmt"]["driven"]}** — under v2
   this was zero, which is what made a credential reset untraceable. Of those,
   {REPORT["mgmt"]["kept_2xx"]} by the management-POST rule and {REPORT["mgmt"]["kept_error"]}
   that returned 4xx/5xx and would have been kept by the error rule whatever rule 5 did — first match wins, and the error rule is matched first, so only the
   first number is evidence about rule 5. Every one, by full path:

{chr(10).join(f"   - `POST {p}` → {s}, **{v}**  ({l})" for l, p, s, v in REPORT["mgmt"]["rows"])}
6. **Volume** — {len(STORED)} records stored for {len(ALL_CALLS)} calls
   ({len(STORED) / max(1, len(ALL_CALLS)):.1f} per call);
   `fluentbit_filter_drop_records_total` delta {DROP_DELTA if DROP_DELTA is not None else "**NOT MEASURED** — " + str(DROP_NOTE)}.
   **Every stored record accounted for:** {VOLUME["attributed"]} attributed to a call in this
   table, {VOLUME["run_other"]} carrying a request id this run minted for a call that is not in
   it, {VOLUME["untagged"]} with no request id at all — {"they sum to the total" if VOLUME["reconciles"] else "**AND THEY DO NOT SUM TO THE TOTAL**"}.
   **{ACCESS_STORED} are access-log records ({100 * ACCESS_STORED / max(1, len(STORED)):.1f}%)
   and {APP_STORED} are application lines**, which ride through untouched by rule 2 — so the
   retention policy governs only that first share of the volume. By logger:

{chr(10).join(f"   - `{l}` x{n}" for l, n in sorted(LOGGERS.items(), key=lambda kv: -kv[1])[:8])}
7. **Client-side latency** (there is no `%D`, so this is the only source):
   median {M.ms.median():.0f} ms, p95 {M.ms.quantile(0.95):.0f} ms, max {M.ms.max():.0f} ms.

## Not callable on this cluster, with the reason

{chr(10).join(f"- **{k}** — {v}" for k, v in NOT_CALLABLE.items())}

## Coverage matrix

```
{M.drop(columns=DISPLAY_DROP).to_string(index=False)}
```

`path` is truncated to its last 58 characters for width; `path_full` is the column every
statistic above reads. `window_start` is the window the call was counted into and
`counted_where` is the report row that actually moved — which, for a 4xx on a resource nobody
read successfully, is `__other__` and not the key in `resource_row`.

## The scheduled flush report

A `dummy` INPUT tagged `polaris.report` ticks every `{TICK_SECONDS}`s into the same Lua filter;
on each `{WINDOW_SECONDS}`s boundary the filter replaces the tick with an ARRAY of records and
resets its counters. **The tick rate is not the report period** -- roughly
{WINDOW_SECONDS // (TICK_SECONDS or 30)} ticks fall inside each window and exactly one report
comes out.

**A run spans several windows and is read as their merge.** At `WINDOW_SECONDS` {WINDOW_SECONDS}
it always does: reading a single window reports whatever landed in the last {WINDOW_SECONDS}
seconds, which on the first attempt was the cleanup DELETEs alone. Schema v1, three record types on one
envelope, on their own stream `{{app="polaris-shipper-report", level="REPORT"}}`.

| | |
|---|---|
| `window_seconds`, as deployed | {WINDOW_SECONDS} |
| windows the run spanned | {len(REPORT["run"]["windows"])} x {WINDOW_SECONDS}s, `{REPORT["run"]["windows"][0]}` .. `{REPORT["run"]["end"]}` (end exclusive) |
| rows per window | {", ".join(f"{w[-9:-1]}={n}" for w, n in sorted(REPORT["run"]["rows_by_window"].items()))} |
| merged rows | {len(REPORT["run"]["merged"])} = 1 summary + {REPORT["recon"]["counts"]["resource"]} resource + {REPORT["recon"]["counts"]["principal"]} principal |
| resource rows this run explains | {REPORT["recon"]["counts"]["resource_expected"]} expected, {len(REPORT["recon"]["unexplained"])} unexplained, {len(REPORT["recon"]["missing"])} missing |
| principals in the report | {len(REPORT["mix"])} |
| per-window invariants | {"all OK" if not any(REPORT["run"]["per_window"].values()) else {k: v for k, v in REPORT["run"]["per_window"].items() if v}} |
| merged invariants | {"OK" if not REPORT["run"]["invariants"] else REPORT["run"]["invariants"]} |
| `resource_kind` values seen | {REPORT["run"]["kinds"]} |
| oracle diff, all fields | {len(REPORT["diff"]["mismatches"])} mismatches |
| oracle diff, THIS run's fixture resources | {len(REPORT["diff"]["ours"])} mismatches{" — **FAILING**" if REPORT["diff"]["ours"] else " — clean"} |
| zero-carry | {len(REPORT["w2"]["carried"])} rows carried at 0 into `{REPORT["w2"]["window"]}` |
| carry decay | {"decayed" if REPORT["w3"]["decayed"] else "STILL CARRIED"} in `{REPORT["w3"]["window"]}` |
| window gaps (a skipped window) | {REPORT["w3"]["gaps"] or "none"} |

The diff excludes the fields that describe the emitting process or which window it was
(`hostname`, `report_seq`, `_msg`, `window_start`/`_end`/`_time`, `partial_window`) plus
`response_bytes`, which no client can predict without `%b`. What remains, top 20:

```
{chr(10).join(f"{m['report_type']:9} {str(m['key'])[-46:]:46} {m['field']:18} oracle={str(m['expected'])[:20]:20} stored={str(m['actual'])[:20]}" for m in _top) or "(none)"}
```

### The fixture's own resource rows — the part that IS asserted

These paths belong to this run alone, so a mismatch here is the pipeline disagreeing with the
deployed Lua rather than a neighbour's traffic. A count is not a finding; every one is named.

```
{chr(10).join(f"{m['report_type']:9} {str(m['key'])[-46:]:46} {m['field']:18} oracle={str(m['expected'])[:20]:20} stored={str(m['actual'])[:20]}" for m in REPORT["diff"]["ours"][:60]) or "(none -- the fixture's rows match the oracle field for field)"}
```

### The assertions PLAN section 7 names

| assertion | | |
|---|---|---|
{chr(10).join(f"| {name} | {'PASS' if ok else ('n/a' if ok is None else '**FAIL**')} | `{detail}` |" for name, ok, detail in REPORT["asserts"])}

### The principal rows, and why more than one of them matters

`sum(principal.requests) == access_seen - parse_errors` is the schema's only real self-check —
but with a SINGLE principal it is satisfied identically by a global counter, so a filter that
attributed nothing at all would still pass it. Run 1 of v3 drove all 132 calls as one identity
and reported the margin OK, which was true and proved nothing.

| principal | requests | reads | writes | errors |
|---|---|---|---|---|
{chr(10).join(f"| `{p}` | {v['requests']} | {v['reads']} | {v['writes']} | {v['errors']} |" for p, v in sorted(REPORT["mix"].items(), key=lambda kv: -kv[1]["requests"]))}

{"Two or more principals carry a strict fraction of the total, so the equality has something to catch." if len([p for p, v in REPORT["mix"].items() if 0 < v["requests"] < sum(x["requests"] for x in REPORT["mix"].values())]) >= 2 else "**One identity carries everything: the margin equality is tautological for this run.**"}

**Mismatches are expected here and are read, not asserted.** The oracle replays `ALL_CALLS`
with each call's measured principal; the real window also holds the fixture setup, every
client's OAuth token exchange (attributed to principal `-`), and any other cluster traffic. The assertion
that matters is the fixture's own resource rows, counted separately above.

The margin equality — `sum(resource.requests) == sum(principal.requests) ==
access_seen - parse_errors` — is the schema's only real self-check, because `errors`
deliberately overlaps `reads`/`writes`. Note that `access_kept + access_counted ==
access_seen` is **tautological in the filter** (`build_report` computes `access_kept` by
subtraction) and is worth asserting only end to end, where it is a transport check.

**What a grant granted survives in one place, and not by policy.** {len(priv_hits)} record(s)
carry the granted privilege name. There is no `%D` and no request body in the access log, so
the only trace of *what* a `grant_privilege` PUT actually granted is an application log line,
which rule 2 passes through untouched. That makes the auditability of grants depend on a
Polaris logger's level rather than on the retention policy under test: the PUT itself is
stored, its content is one configuration change from disappearing, and nothing here protects
it. v3 closed "who created a principal"; "what privilege was granted" is still open.

**A startup blind spot, measured:** `report_tick` opens its first window on the FIRST tick, and
`count_record()` returns immediately while `counts` is nil. Records processed between shipper
start and that first tick are routed by the policy correctly but appear in NO report. Bounded
by the tick interval ({TICK_SECONDS}s as deployed). The window they land in is flagged
`partial_window: true`.
"""
out = pathlib.Path("doc-log-coverage-results.md")
out.write_text(report, encoding="utf-8")
print(f"  wrote {out} ({len(report):,} bytes)")
print("\n  Next: update MEMORY.md 'Now', put the numbers in .memory/roadmap.md, anything")
print("  not to be trusted in .memory/active-issues.md, and the wrong turns in")
print("  .memory/sessions/<date>-log-coverage.md. Then commit -- one task, one commit.")

  --- stored records by loggerName ---
     1521  org.apache.polaris.persistence.relational.jdbc.DatasourceOperations
      250  org.apache.polaris.service.catalog.iceberg.IcebergCatalog
      172  org.apache.polaris.service.catalog.io.StorageAccessConfigProvider
       93  org.apache.polaris.core.storage.cache.StorageCredentialCache
       80  org.apache.polaris.core.persistence.resolver.PolarisResolutionManifest
       78  org.apache.polaris.service.catalog.iceberg.IcebergCatalogHandler
       58  org.apache.iceberg.CatalogUtil
       56  io.quarkus.http.access-log
       41  org.apache.polaris.service.exception.IcebergExceptionMapper
       41  org.apache.iceberg.BaseMetastoreCatalog
       37  org.apache.polaris.service.catalog.iceberg.IcebergCatalogAdapter
       28  org.apache.polaris.core.storage.LocationRestrictions
  access-log records: 56 of 2500 (2.2%) -- the filter governs only these
  wrote doc-log-coverage-results.md (81,236 bytes)

  Next: update MEMORY.md 'Now', put the